<a href="https://colab.research.google.com/github/ngcyber-sys/ML_CU_-earthquakes/blob/main/ML_%D0%97%D0%B5%D0%BC%D0%BB%D0%B5%D1%82%D1%80%D1%8F%D1%81%D0%B5%D0%BD%D0%B8%D1%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Анализ землетрясений

**Цель:** Понять структуру данных и выявить закономерности, которые впоследствии помогут предсказывать, будет ли землетрясение достаточно сильным - **магнитуда >= 4.5** (порог, после которого событие ощущается людьми и может причинить ущерб).

**Источник данных:** USGS Earthquake API

---

## Содержание
1. [Загрузка и первичный осмотр](#1)
2. [Качество данных](#2)
3. [Целевая переменная: сильное vs слабое](#3)
4. [Распределение магнитуды](#4)
5. [Глубина залегания](#5)
6. [Географический анализ](#6)
7. [Временной анализ](#7)
8. [Корреляции и взаимосвязи признаков](#8)
9. [Региональный анализ](#9)
9б. [Координаты × Глубина: тектонические закономерности](#9b)
9в. [b-value: региональный наклон закона Гутенберга–Рихтера](#9v)
10. [Пространственно-временные признаки](#10)
10б. [Форшоки: предшествующая активность](#10b)
11. [Итоговый отчёт](#11)
12. [Baseline](#12)
13. [Итог по аномалиям и выбросам](#13)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from datasets import load_dataset

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_palette('husl')

STRONG_THRESHOLD = 4.5

<a id='1'></a>
## 1. Загрузка и первичный осмотр

In [ ]:
ds = load_dataset("leorigasaki54/earthquake-events-180d")
df = pd.DataFrame(ds["train"])

df['time_utc'] = pd.to_datetime(df['time_utc'])
print(f'Форма датасета: {df.shape}')
print(f'\nТипы данных:')
print(df.dtypes)
df.head(10)

In [ ]:
print('Базовая статистика')
df[['mag', 'depth_km', 'latitude', 'longitude']].describe().round(3)

<a id='2'></a>
## 2. Качество данных

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
quality_df = pd.DataFrame({'Пропуски': missing, '%': missing_pct})
print('Пропущенные значения')
print(quality_df)

dups = df.duplicated(subset='id').sum()
print(f'\nДублей по id: {dups}')

print(f'\nПериод данных: {df.time_utc.min().date()} - {df.time_utc.max().date()}')
print(f'Дней охвата: {(df.time_utc.max() - df.time_utc.min()).days}')

In [ ]:
neg_depth = (df.depth_km < 0).sum()
neg_mag   = (df.mag < 0).sum()
zero_mag  = (df.mag == 0).sum()

print(f'Отрицательная глубина: {neg_depth} ({neg_depth/len(df)*100:.1f}%)')
print(f'Отрицательная магнитуда: {neg_mag} ({neg_mag/len(df)*100:.1f}%)')
print(f'Нулевая магнитуда: {zero_mag} ({zero_mag/len(df)*100:.1f}%)')
print()
print('Отрицательная глубина - артефакт алгоритма гипоцентральной локации USGS:')
print('означает «очень близко к поверхности» (погрешность модели, а не реальная глубина).')
print('Удалять не нужно - при необходимости округлить до 0: df.depth_km.clip(lower=0).')
print()
print('Отрицательные и нулевые магнитуды - корректные физические значения.')
print('Шкала магнитуд логарифмическая и не имеет нижней границы.')
print('Современные сейсмографы фиксируют события, невидимые старым приборам.')

In [ ]:
df['is_strong'] = (df['mag'] >= STRONG_THRESHOLD).astype(int)
df['hour']      = df['time_utc'].dt.hour
df['dayofweek'] = df['time_utc'].dt.dayofweek
df['month']     = df['time_utc'].dt.month
df['date']      = df['time_utc'].dt.date

df['region'] = df['place'].str.extract(r'of (.+)$')[0].str.split(',').str[-1].str.strip()
df['region'] = df['region'].fillna(df['place'])

<a id='3'></a>
## 3. Целевая переменная: сильное vs слабое землетрясение

In [ ]:
counts = df['is_strong'].value_counts()
labels = ['Слабое (mag < 4.5)', 'Сильное (mag >= 4.5)']
colors = ['#4A90D9', '#E74C3C']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].pie(
    counts.values, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 12}
)
axes[0].set_title('Распределение классов', fontsize=14, fontweight='bold', pad=15)

bars = axes[1].bar(labels, counts.values, color=colors, width=0.5, edgecolor='white')
for bar, val in zip(bars, counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}', ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Количество событий', fontsize=12)
axes[1].set_title('Абсолютные числа', fontsize=14, fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Дисбаланс классов', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

ratio = counts[0] / counts[1]
print(f'Соотношение классов: {ratio:.1f}:1 (слабые:сильные)')
print(f'Доля сильных событий: {counts[1]/len(df)*100:.2f}%')

**Вывод:** Классы сильно несбалансированы - сильные землетрясения составляют лишь  5% от всех событий. При обучении моделей потребуется использовать техники балансировки.

<a id='4'></a>
## 4. Распределение магнитуды

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['mag'], bins=80, color='#4A90D9', edgecolor='white', alpha=0.85)
axes[0].axvline(STRONG_THRESHOLD, color='#E74C3C', linewidth=2.5,
                linestyle='--', label=f'Порог {STRONG_THRESHOLD}')
axes[0].axvline(df['mag'].mean(), color='#2ECC71', linewidth=2,
                linestyle=':', label=f'Среднее {df["mag"].mean():.2f}')
axes[0].axvline(df['mag'].median(), color='#F39C12', linewidth=2,
                linestyle=':', label=f'Медиана {df["mag"].median():.2f}')
axes[0].set_xlabel('Магнитуда', fontsize=12)
axes[0].set_ylabel('Частота', fontsize=12)
axes[0].set_title('Распределение магнитуды (все события)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)

df_box = df.copy()
df_box['Класс'] = df_box['is_strong'].map({0: 'Слабое (< 4.5)', 1: 'Сильное (>= 4.5)'})
bp = axes[1].boxplot(
    [df[df.is_strong==0]['mag'], df[df.is_strong==1]['mag']],
    labels=['Слабое\n(< 4.5)', 'Сильное\n(>= 4.5)'],
    patch_artist=True,
    boxprops=dict(facecolor='#4A90D9', alpha=0.7),
    medianprops=dict(color='#E74C3C', linewidth=2.5),
    flierprops=dict(marker='o', alpha=0.3, markersize=3)
)
bp['boxes'][1].set_facecolor('#E74C3C')
axes[1].set_ylabel('Магнитуда', fontsize=12)
axes[1].set_title('Boxplot магнитуды по классам', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print('Статистика магнитуды по классам:')
print(df.groupby('is_strong')['mag'].describe().round(3))

In [ ]:
mag_bins = np.arange(df.mag.min() // 0.5 * 0.5, df.mag.max() + 0.5, 0.5)
counts_gr, edges = np.histogram(df['mag'], bins=mag_bins)
cumulative = np.cumsum(counts_gr[::-1])[::-1]
bin_centers = (edges[:-1] + edges[1:]) / 2

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(bin_centers, cumulative, 'o-', color='#4A90D9',
            linewidth=2, markersize=4, label='Наблюдаемые данные')
ax.axvline(STRONG_THRESHOLD, color='#E74C3C', linewidth=2.5,
           linestyle='--', label=f'Порог {STRONG_THRESHOLD}')
ax.set_xlabel('Магнитуда', fontsize=12)
ax.set_ylabel('Кумулятивное число событий (log)', fontsize=12)
ax.set_title('Закон Гутенберга–Рихтера: N(>=M) по магнитуде', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('График подтверждает закон Гутенберга–Рихтера:')
print('число землетрясений экспоненциально убывает с ростом магнитуды.')

<a id='5'></a>
## 5. Глубина залегания

In [ ]:
def depth_category(d):
    if d < 0:   return 'Аномальная (< 0)'
    if d <= 70: return 'Мелкофокусное (0–70)'
    if d <= 300: return 'Среднефокусное (70–300)'
    return 'Глубокофокусное (> 300)'

df['depth_cat'] = df['depth_km'].apply(depth_category)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_valid = df[df.depth_km >= 0]
axes[0].hist(df_valid['depth_km'], bins=80, color='#8E44AD', edgecolor='white', alpha=0.85)
for limit, label in [(70, '70 км'), (300, '300 км')]:
    axes[0].axvline(limit, color='#E74C3C', linewidth=1.8, linestyle='--', label=label)
axes[0].set_xlabel('Глубина (км)', fontsize=12)
axes[0].set_ylabel('Частота', fontsize=12)
axes[0].set_title('Распределение глубины (depth_km >= 0)', fontsize=13, fontweight='bold')
axes[0].legend(title='Границы категорий', fontsize=10)

cat_counts = df['depth_cat'].value_counts()
colors_cat = ['#3498DB', '#27AE60', '#E67E22', '#E74C3C']
wedges, texts, autotexts = axes[1].pie(
    cat_counts.values, labels=cat_counts.index,
    colors=colors_cat[:len(cat_counts)],
    autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 9}
)
axes[1].set_title('Распределение по категориям глубины', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

weak   = df[df.is_strong == 0]
strong = df[df.is_strong == 1]

ax.scatter(weak['depth_km'],   weak['mag'],   alpha=0.15, s=8,
           color='#4A90D9', label='Слабое (< 4.5)', rasterized=True)
ax.scatter(strong['depth_km'], strong['mag'], alpha=0.6,  s=15,
           color='#E74C3C', label='Сильное (>= 4.5)', zorder=5)

ax.axhline(STRONG_THRESHOLD, color='black', linewidth=1.5, linestyle='--', alpha=0.5)
ax.set_xlabel('Глубина (км)', fontsize=12)
ax.set_ylabel('Магнитуда', fontsize=12)
ax.set_title('Зависимость магнитуды от глубины', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, markerscale=3)
ax.set_xlim(-10, 700)
plt.tight_layout()
plt.show()

print('Глубина по классам:')
print(df.groupby('is_strong')['depth_km'].describe().round(2))

<a id='6'></a>
## 6. Географический анализ

In [ ]:
fig = px.scatter_geo(
    df,
    lat='latitude', lon='longitude',
    color='mag',
    color_continuous_scale=[
        (0.0, '#3498DB'), (0.5, '#F39C12'), (0.8, '#E74C3C'), (1.0, '#8E1010')
    ],
    size=np.where(df['mag'] >= STRONG_THRESHOLD,
                  np.clip(df['mag'] - 3, 1, 8)**2,
                  1.5),
    size_max=18,
    hover_name='place',
    hover_data={'mag': ':.2f', 'depth_km': ':.1f', 'time_utc': True,
                'latitude': False, 'longitude': False},
    title='Карта землетрясений (180 дней, размер = магнитуда для mag >= 4.5)',
    projection='natural earth',
    template='plotly_white',
    height=550
)
fig.update_layout(
    coloraxis_colorbar=dict(title='Магнитуда'),
    margin=dict(l=0, r=0, t=50, b=0)
)
fig.show()

In [ ]:
df_strong = df[df.is_strong == 1].copy()

fig = px.scatter_geo(
    df_strong.sort_values('mag'),
    lat='latitude', lon='longitude',
    color='mag',
    size=np.clip(df_strong.sort_values('mag')['mag'] - 3, 1, 10) ** 1.7,
    size_max=25,
    color_continuous_scale='OrRd',
    hover_name='place',
    hover_data={'mag': ':.2f', 'depth_km': ':.1f', 'time_utc': True},
    title=f'Сильные землетрясения (mag >= {STRONG_THRESHOLD}) - {len(df_strong)} событий',
    projection='natural earth',
    template='plotly_white',
    height=500
)
fig.show()

print(f'\nТоп-10 самых сильных событий:')
cols = ['time_utc', 'mag', 'depth_km', 'place', 'latitude', 'longitude']
print(df_strong.nlargest(10, 'mag')[cols].to_string(index=False))

<a id='7'></a>
## 7. Временной анализ

In [ ]:
daily = df.groupby('date').agg(
    total=('id', 'count'),
    strong=('is_strong', 'sum'),
    avg_mag=('mag', 'mean')
).reset_index()
daily['date'] = pd.to_datetime(daily['date'])

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].fill_between(daily['date'], daily['total'], alpha=0.4, color='#4A90D9')
axes[0].plot(daily['date'], daily['total'], color='#4A90D9', linewidth=1.2)
axes[0].set_ylabel('Всего событий / день', fontsize=11)
axes[0].set_title('Сейсмическая активность по дням', fontsize=13, fontweight='bold')
roll = daily['total'].rolling(7, center=True).mean()
axes[0].plot(daily['date'], roll, color='#E74C3C', linewidth=2.5,
             label='7-дн. среднее', zorder=5)
axes[0].legend()

axes[1].bar(daily['date'], daily['strong'], color='#E74C3C', alpha=0.8, width=1)
axes[1].set_ylabel('Сильных событий / день', fontsize=11)
axes[1].set_xlabel('Дата', fontsize=11)
axes[1].set_title('Число сильных землетрясений (mag >= 4.5) в сутки', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
hour_counts  = df.groupby('hour')['is_strong'].agg(['count','sum']).reset_index()
hour_counts['strong_pct'] = hour_counts['sum'] / hour_counts['count'] * 100

dow_counts = df.groupby('dayofweek')['is_strong'].agg(['count','sum']).reset_index()
dow_counts['strong_pct'] = dow_counts['sum'] / dow_counts['count'] * 100
dow_labels = ['Пн','Вт','Ср','Чт','Пт','Сб','Вс']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(hour_counts['hour'], hour_counts['count'], color='#4A90D9', alpha=0.7, label='Все события')
ax2 = axes[0].twinx()
ax2.plot(hour_counts['hour'], hour_counts['strong_pct'], 'o-',
         color='#E74C3C', linewidth=2, markersize=5, label='% сильных')
ax2.set_ylabel('Доля сильных, %', fontsize=11, color='#E74C3C')
ax2.tick_params(colors='#E74C3C')
axes[0].set_xlabel('Час UTC', fontsize=11)
axes[0].set_ylabel('Число событий', fontsize=11)
axes[0].set_title('Распределение по часам суток', fontsize=13, fontweight='bold')
axes[0].set_xticks(range(0, 24, 2))

axes[1].bar(dow_counts['dayofweek'], dow_counts['count'], color='#8E44AD', alpha=0.7)
ax3 = axes[1].twinx()
ax3.plot(dow_counts['dayofweek'], dow_counts['strong_pct'], 's-',
         color='#E74C3C', linewidth=2, markersize=6, label='% сильных')
ax3.set_ylabel('Доля сильных, %', fontsize=11, color='#E74C3C')
ax3.tick_params(colors='#E74C3C')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(dow_labels)
axes[1].set_xlabel('День недели', fontsize=11)
axes[1].set_ylabel('Число событий', fontsize=11)
axes[1].set_title('Распределение по дням недели', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print('Вывод: данные не показывают значимой зависимости от часа суток или дня недели.')
print('Небольшие колебания доли сильных событий статистически незначимы при n 20000.')
print('Это согласуется с сейсмологией: возникновение землетрясений не привязано ко времени суток.')

<a id='8'></a>
## 8. Корреляции и взаимосвязи признаков

In [ ]:
num_cols = ['mag', 'depth_km', 'latitude', 'longitude', 'hour', 'is_strong']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.3f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Корреляция Пирсона'}
)
ax.set_title('Корреляционная матрица признаков', fontsize=14, fontweight='bold', pad=15)
col_labels = ['Магнитуда', 'Глубина', 'Широта', 'Долгота', 'Час UTC', 'Целевой']
ax.set_xticklabels(col_labels, rotation=30, ha='right')
ax.set_yticklabels(col_labels, rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
features = ['depth_km', 'latitude', 'longitude']
titles   = ['Глубина (км)', 'Широта', 'Долгота']
colors   = ['#4A90D9', '#E74C3C']

for ax, feat, title in zip(axes, features, titles):
    data_0 = df[df.is_strong == 0][feat].dropna()
    data_1 = df[df.is_strong == 1][feat].dropna()

    parts0 = ax.violinplot(data_0, positions=[0], showmedians=True)
    parts1 = ax.violinplot(data_1, positions=[1], showmedians=True)

    for pc, color in zip(
        [parts0['bodies'][0], parts1['bodies'][0]], colors
    ):
        pc.set_facecolor(color)
        pc.set_alpha(0.7)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Слабое', 'Сильное'], fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(f'{title} по классам', fontsize=12, fontweight='bold')

plt.suptitle('Сравнение распределений признаков по классам', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
df_plot = df[(df.depth_km >= 0) & (df.depth_km <= 400)]

fig, ax = plt.subplots(figsize=(11, 6))
h = ax.hist2d(
    df_plot['depth_km'], df_plot['mag'],
    bins=[60, 60], cmap='YlOrRd'
)
plt.colorbar(h[3], ax=ax, label='Число событий')
ax.axhline(STRONG_THRESHOLD, color='blue', linewidth=2, linestyle='--',
           label=f'Порог mag = {STRONG_THRESHOLD}')
ax.set_xlabel('Глубина (км)', fontsize=12)
ax.set_ylabel('Магнитуда', fontsize=12)
ax.set_title('2D плотность: Глубина × Магнитуда', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

<a id='9'></a>
## 9. Региональный анализ

In [ ]:
top_regions = df['region'].value_counts().head(12)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors_bar = sns.color_palette('husl', len(top_regions))
bars = axes[0].barh(top_regions.index[::-1], top_regions.values[::-1],
                    color=colors_bar, edgecolor='white')
for bar, val in zip(bars, top_regions.values[::-1]):
    axes[0].text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=9)
axes[0].set_xlabel('Число событий', fontsize=11)
axes[0].set_title('Топ-12 регионов по числу событий', fontsize=13, fontweight='bold')

region_stats = df.groupby('region').agg(
    total=('id','count'),
    strong=('is_strong','sum')
).query('total >= 50')
region_stats['strong_pct'] = region_stats['strong'] / region_stats['total'] * 100
top_risky = region_stats.nlargest(12, 'strong_pct')

palette = sns.color_palette('Reds_r', len(top_risky))
bars2 = axes[1].barh(top_risky.index[::-1], top_risky['strong_pct'][::-1],
                     color=palette, edgecolor='white')
for bar, val in zip(bars2, top_risky['strong_pct'][::-1]):
    axes[1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)
axes[1].set_xlabel('% сильных событий (mag >= 4.5)', fontsize=11)
axes[1].set_title('Топ-12 регионов по % сильных событий', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
top10 = df['region'].value_counts().head(10).index.tolist()
df_top = df[df['region'].isin(top10)]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

order = df_top.groupby('region')['mag'].median().sort_values(ascending=False).index

sns.boxplot(data=df_top, x='region', y='mag', order=order,
            palette='husl', ax=axes[0])
axes[0].axhline(STRONG_THRESHOLD, color='red', linewidth=2, linestyle='--',
                label=f'Порог {STRONG_THRESHOLD}')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right', fontsize=9)
axes[0].set_xlabel('')
axes[0].set_ylabel('Магнитуда', fontsize=11)
axes[0].set_title('Распределение магнитуды по топ-10 регионам', fontsize=12, fontweight='bold')
axes[0].legend()

sns.boxplot(data=df_top, x='region', y='depth_km', order=order,
            palette='husl', ax=axes[1])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right', fontsize=9)
axes[1].set_xlabel('')
axes[1].set_ylabel('Глубина (км)', fontsize=11)
axes[1].set_title('Распределение глубины по топ-10 регионам', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

<a id='9b'></a>
## 9б. Координаты × Глубина: тектонические закономерности

**Идея:** В зонах субдукции (Тихоокеанское огненное кольцо) плита уходит под углом вглубь. Чем дальше от желоба - тем глубже очаги. Это называется **зоной Вадати–Бениоффа**. Комбинация `(широта, долгота, глубина)` несёт информацию о тектонической обстановке и может быть значимым предиктором силы события.

In [ ]:
# Только корректные глубины
df_v = df[df['depth_km'] >= 0].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, coord, label in [
    (axes[0], 'latitude',  'Широта'),
    (axes[1], 'longitude', 'Долгота'),
]:
    weak   = df_v[df_v.is_strong == 0]
    strong = df_v[df_v.is_strong == 1]
    ax.scatter(weak[coord],   weak['depth_km'],   alpha=0.08, s=5,
               color='#4A90D9', rasterized=True, label='Слабое')
    ax.scatter(strong[coord], strong['depth_km'], alpha=0.5,  s=15,
               color='#E74C3C', zorder=5, label='Сильное (>= 4.5)')
    ax.invert_yaxis()
    ax.set_xlabel(label, fontsize=12)
    ax.set_ylabel('Глубина (км)', fontsize=12)
    ax.set_title(f'{label} × Глубина', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10, markerscale=3)

plt.suptitle('Зависимость глубины от координат (ось Y - вглубь земли)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
df_plot = df[df['depth_km'] >= 0].copy()
df_plot['depth_log'] = np.log1p(df_plot['depth_km'])

fig = px.scatter_geo(
    df_plot.sample(min(5000, len(df_plot)), random_state=42),
    lat='latitude', lon='longitude',
    color='depth_km',
    color_continuous_scale=[
        (0.0,  '#FFFFCC'),
        (0.15, '#A1DAB4'),
        (0.4,  '#41B6C4'),
        (0.7,  '#2C7FB8'),
        (1.0,  '#253494'),
    ],
    size_max=4,
    hover_name='place',
    hover_data={'depth_km': ':.1f', 'mag': ':.2f'},
    title='Карта глубин: чем темнее - тем глубже очаг',
    projection='natural earth',
    template='plotly_white',
    height=520
)
fig.update_traces(marker_size=3)
fig.update_layout(coloraxis_colorbar=dict(title='Глубина (км)'))
fig.show()

In [ ]:
# Зона Вадати–Бениоффа: срезы по трём крупным субдукционным зонам.
# Координаты в системе −180..+180 (стандарт USGS).
# Ось X выбирается перпендикулярно желобу:
#   Япония/Курилы - желоб N–S, плита уходит на запад -> ось долготы
#   Чили/Перу     - желоб N–S, плита уходит на восток -> ось долготы
#   Алеуты/Аляска - желоб E–W, плита уходит на север -> ось широты
zones = {
    'Япония / Курилы': dict(lon=(125, 155),  lat=(20, 60),  axis='longitude', xlabel='Долгота'),
    'Чили / Перу':     dict(lon=(-80, -65),  lat=(-55,  0), axis='longitude', xlabel='Долгота'),
    'Алеуты / Аляска': dict(lon=(-180, -140), lat=(48, 62), axis='latitude',  xlabel='Широта'),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (title, z) in zip(axes, zones.items()):
    mask = (
        (df['longitude'] >= z['lon'][0]) & (df['longitude'] <= z['lon'][1]) &
        (df['latitude']  >= z['lat'][0]) & (df['latitude']  <= z['lat'][1]) &
        (df['depth_km']  >= 0)
    )
    sub = df[mask]
    if len(sub) < 5:
        ax.set_title(f'{title}\n(мало данных)', fontsize=10)
        continue

    coord = z['axis']
    weak   = sub[sub.is_strong == 0]
    strong = sub[sub.is_strong == 1]
    ax.scatter(weak[coord],   weak['depth_km'],   alpha=0.2, s=8,
               color='#4A90D9', rasterized=True, label='Слабое')
    ax.scatter(strong[coord], strong['depth_km'], alpha=0.8, s=25,
               color='#E74C3C', zorder=5, label='Сильное')
    ax.invert_yaxis()
    ax.set_xlabel(z['xlabel'], fontsize=11)
    ax.set_ylabel('Глубина (км)', fontsize=11)
    ax.set_title(f'{title}\n(n={len(sub):,})', fontsize=10, fontweight='bold')
    ax.legend(fontsize=9, markerscale=2)

plt.suptitle('Зона Вадати–Бениоффа: наклонные слои в зонах субдукции',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Вывод: Зона Вадати–Бениоффа

Все три зоны подтверждают существование **наклонных сейсмических поясов** -
признак субдукции океанической плиты под континентальную.

| Регион | Глубина | Особенности |
|--------|---------|-------------|
| Япония / Курилы | до 500 км | сильные события на 0–150 км |
| Чили / Перу | до 300 км | самая чёткая зона субдукции |
| Алеуты / Аляска | до 250 км | большинство событий мелкие (0–50 км) |

> **Главный вывод:** сильные события (M >= 4.5) концентрируются на глубинах
> до 150 км, где напряжения в плите максимальны.

In [ ]:
df_v = df[df['depth_km'] >= 0].copy()
df_v['lat_zone'] = pd.cut(
    df_v['latitude'],
    bins=[-90, -60, -30, -10, 10, 30, 50, 70, 90],
    labels=['90–60°S','60–30°S','30–10°S','10°S–10°N',
            '10–30°N','30–50°N','50–70°N','70–90°N']
)

pivot = df_v.pivot_table(
    index='depth_cat', columns='lat_zone',
    values='is_strong', aggfunc='mean'
) * 100

depth_order = ['Аномальная (< 0)', 'Мелкофокусное (0–70)',
               'Среднефокусное (70–300)', 'Глубокофокусное (> 300)']
pivot = pivot.reindex([d for d in depth_order if d in pivot.index])

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    pivot, annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.5, ax=ax, cbar_kws={'label': '% сильных событий'},
    mask=pivot.isnull()
)
ax.set_xlabel('Широтный пояс', fontsize=11)
ax.set_ylabel('Категория глубины', fontsize=11)
ax.set_title('% сильных событий (mag >= 4.5) по глубине и широтному поясу',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Тёмные клетки = зоны, где сильные землетрясения встречаются чаще всего.')

In [ ]:
df_3d = df[df['depth_km'] >= 0].sample(min(4000, len(df)), random_state=42).copy()

fig = px.scatter_3d(
    df_3d,
    x='longitude', y='latitude', z='depth_km',
    color='mag',
    color_continuous_scale=[
        (0.0, '#3498DB'), (0.6, '#F39C12'),
        (0.85, '#E74C3C'), (1.0, '#7B0000')
    ],
    size=np.where(df_3d['is_strong'] == 1, 4, 1.5),
    size_max=8,
    opacity=0.6,
    hover_name='place',
    hover_data={'mag': ':.2f', 'depth_km': ':.1f'},
    title='3D: Долгота × Широта × Глубина (цвет = магнитуда)',
    labels={'longitude': 'Долгота', 'latitude': 'Широта', 'depth_km': 'Глубина (км)'},
    height=620
)
fig.update_layout(
    scene=dict(
        zaxis=dict(autorange='reversed', title='Глубина (км)'),
        camera=dict(eye=dict(x=1.5, y=1.5, z=0.7))
    ),
    coloraxis_colorbar=dict(title='Магнитуда')
)
fig.show()

In [ ]:
# Инженерные признаки на основе координат + глубины

# 1. Широтные зоны: целочисленные коды 0…7 (labels=False)
# Допущение: равные интервалы между зонами - приемлемно для EDA, но не для финальной модели
df['lat_zone_num'] = pd.cut(
    df['latitude'],
    bins=[-90, -60, -30, -10, 10, 30, 50, 70, 90],
    labels=False
).astype(float)

# 2. Расстояние от экватора (абс. широта)
df['abs_latitude'] = df['latitude'].abs()

# 3. Взаимодействие глубины и широты
df['depth_x_abs_lat'] = df['depth_km'] * df['abs_latitude']

# 4. Категориальная глубина (уже есть depth_cat) - числовой код
depth_map = {
    'Аномальная (< 0)':       0,
    'Мелкофокусное (0–70)':   1,
    'Среднефокусное (70–300)':2,
    'Глубокофокусное (> 300)':3,
}
df['depth_cat_num'] = df['depth_cat'].map(depth_map)

# 5. Взаимодействие категории глубины и широтного пояса
df['depth_lat_interact'] = df['depth_cat_num'] * df['lat_zone_num']

eng_feats = ['abs_latitude','depth_x_abs_lat','depth_cat_num',
             'lat_zone_num','depth_lat_interact']
corr = df[eng_feats + ['is_strong']].corr()['is_strong'].drop('is_strong')
corr = corr.sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors_bar = ['#E74C3C' if v > 0 else '#4A90D9' for v in corr.values]
ax.barh(corr.index[::-1], corr.values[::-1], color=colors_bar[::-1],
        edgecolor='white', height=0.55)
ax.axvline(0, color='black', linewidth=1)
for i, (idx, val) in enumerate(zip(corr.index[::-1], corr.values[::-1])):
    ax.text(val + (0.001 if val >= 0 else -0.001), i,
            f'{val:.3f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=10)
ax.set_xlabel('Корреляция с is_strong', fontsize=11)
ax.set_title('Инженерные признаки: координаты + глубина', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(corr.to_string())

**Выводы:**

**1. Мелкофокусные события (0–70 км) - везде, глубокие - только в субдукционных зонах**

Мелкие землетрясения встречаются у всех типов разломов: трансформных, срединно-океанических хребтов и субдукционных желобов. Среднефокусные (70–300 км) и глубокофокусные (> 300 км) события практически полностью ограничены зонами субдукции - на глобальной карте они образуют узкие полосы Тихоокеанского огненного кольца. Это фундаментальная географическая закономерность: глубина > 70 км сама по себе является косвенным признаком субдукционной обстановки.

**2. Зона Вадати–Бениоффа - наклонный сейсмический слой**

На региональных срезах (перпендикулярно желобу) чётко видна наклонная зона сейсмичности:
- **Япония / Курилы:** глубина нарастает с востока на запад (Тихоокеанская плита уходит под Евразийскую)
- **Чили / Перу:** глубина нарастает с запада на восток (плита Наска уходит под Южную Америку)
- **Алеуты / Аляска:** глубина нарастает с юга на север (Тихоокеанская плита уходит под Северо-Американскую)

Угол наклона зоны варьируется по регионам ( 30–70°), что определяет профиль глубин на срезе.

**3. Широта × глубина: где чаще встречаются сильные события**

Тепловая карта (ячейка выше) показывает: на тропических и субтропических широтах (10°S–30°N) в сочетании со среднефокусными глубинами доля сильных событий заметно выше. Это Индонезия, Филиппины, Папуа–Новая Гвинея, Тонга - самая активная часть Тихоокеанского кольца. Сочетание «тропическая широта + глубина 70–300 км» - потенциально информативный признак для модели.

| Наблюдение | Вывод |
|---|---|
| Глубокие события строго привязаны к зонам субдукции |  работает как косвенный детектор субдукционной обстановки |
| Зона Вадати–Бениоффа видна на региональных срезах | Глубина нарастает перпендикулярно желобу; направление субдукции определяет, по какой оси строить срез |
| Тропические широты + среднефокусные глубины | Повышенная доля сильных событий - зоны активной субдукции |
| Глубокофокусные (> 300 км) редки в датасете ( 1%) | Статистика нестабильна; в реальности могут быть очень сильными (Боливия 1994: M 8.2 на 631 км) |
| , широтно-глубинная сетка | Несут реальный сигнал; корреляция с  умеренная, но выше, чем у каждого признака по отдельности |

> **Ограничение:** координаты - косвенный признак тектонической зоны. Явное расстояние до ближайшей границы плит (GeoJSON USGS Tectonic Plates) повысило бы интерпретируемость и точность.

<a id='9v'></a>
## 9в. b-value: региональный наклон закона Гутенберга–Рихтера

**Идея:** b-value характеризует *соотношение числа слабых и сильных* землетрясений в регионе. Значение b = 1.0 - глобальная норма.
- **b < 0.9:** высокие тектонические напряжения, больше сильных событий (зоны субдукции, «запертые» разломы)
- **b > 1.1:** дисперсная или вулканическая сейсмичность, относительно меньше сильных событий

**Метод оценки:** максимального правдоподобия (Aki, 1965):
b = log₁₀(e) / (mean(M) − Mc), где Mc - магнитуда полноты каталога.

In [ ]:
mag_complete = 2.5

top10_regions = df['region'].value_counts().head(10).index.tolist()

b_rows = []
for region in top10_regions:
    sub = df[(df['region'] == region) & (df['mag'] >= mag_complete)]
    n = len(sub)
    if n < 20:
        continue
    mean_mag = sub['mag'].mean()
    b_val = np.log10(np.e) / (mean_mag - mag_complete)   # MLE Aki 1965
    b_err = b_val / np.sqrt(n)
    b_rows.append({'region': region,
                   'b_value': round(b_val, 3),
                   'b_err':   round(b_err, 3),
                   'n_events': n})

b_df = pd.DataFrame(b_rows).sort_values('b_value').reset_index(drop=True)

def b_color(b):
    if b < 0.9:  return '#E74C3C'   # красный - опасно
    if b < 1.1:  return '#F39C12'   # оранжевый - норма
    return '#27AE60'                 # зелёный - мало сильных

colors_b = [b_color(v) for v in b_df['b_value']]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(b_df['region'], b_df['b_value'],
               color=colors_b, alpha=0.85,
               xerr=b_df['b_err'], capsize=4,
               error_kw={'linewidth': 1.5, 'ecolor': 'gray'},
               edgecolor='white', label='b ± σ')
ax.axvline(1.0, color='gray', linewidth=2, linestyle='--',
           alpha=0.7, label='b = 1.0 (мировая норма)')

for i, (bar, n) in enumerate(zip(bars, b_df['n_events'])):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'n={n}', va='center', fontsize=9, color='gray')
ax.set_xlabel('b-value (наклон закона Гутенберга–Рихтера)', fontsize=11)
ax.set_title(f'b-value по топ-10 регионам (Mc = {mag_complete})',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(axis='x', alpha=0.3)
ax.set_xlim(0, max(b_df['b_value']) + 0.35)
plt.tight_layout()
plt.show()

print("\nb-value по регионам:")
print(b_df[['region', 'b_value', 'b_err', 'n_events']].to_string(index=False))

**Выводы по b-value:**

| Группа | Регионы | Интерпретация |
|---|---|---|
| b < 0.9 (красные) | Зоны субдукции и запертых разломов | Высокие напряжения -> повышенная доля сильных событий относительно слабых |
| b = 1.0 (оранжевые) | Типичные коровые разломы | Соответствует глобальной норме Гутенберга–Рихтера |
| b > 1.1 (зелёные) | Дисперсная/вулканическая сейсмичность | Меньше крупных событий относительно мелких |

**Ограничения оценки:**
- b-value чувствительно к выбору Mc: при заниженном Mc оценка b завышается
- При малом n (< 50) погрешность σ_b > 0.1 - результат нестабилен
- Mc варьируется по регионам: в удалённых районах USGS регистрирует только mag > 3–4

> **Для ML:** b-value как признак на уровне пространственной ячейки (не региона) потенциально полезен, но требует достаточной истории событий. На 180-дневных данных - ориентировочная надёжность.

<a id='10'></a>
## 10. Пространственно-временные признаки (сейсмический контекст)

**Идея:** Землетрясения не независимы - после крупного толчка следуют афтершоки (закон Омори: частота убывает как 1/t). Знание о *прошлой активности* в точке улучшает прогноз.

**Без утечки данных:** датасет сортируется по времени, и для каждого события признаки считаются только по событиям, произошедшим **строго раньше** него.

In [ ]:
df = df.sort_values('time_utc').reset_index(drop=True)
df = df.set_index('time_utc')

for label, window in [('1h','1h'), ('6h','6h'), ('24h','24h'), ('7d','7D')]:
    roll = df['mag'].rolling(window, closed='left', min_periods=0)
    df[f'global_cnt_{label}']      = roll.count()
    df[f'global_max_mag_{label}']  = roll.max()
    df[f'global_mean_mag_{label}'] = roll.mean()

df = df.reset_index()
print('Глобальные rolling-признаки добавлены')
df[['time_utc','mag','global_cnt_1h','global_cnt_24h','global_max_mag_24h']].head(6)

In [ ]:
df['lat_grid'] = (df['latitude']  / 2).round() * 2
df['lon_grid'] = (df['longitude'] / 2).round() * 2
df['grid_cell'] = df['lat_grid'].astype(str) + '_' + df['lon_grid'].astype(str)

n = len(df)
local_cnt_24h     = np.zeros(n)
local_max_24h     = np.full(n, np.nan)
local_cnt_7d      = np.zeros(n)
local_time_since  = np.full(n, np.nan)

TD_24H = np.timedelta64(24, 'h')
TD_7D  = np.timedelta64(7*24, 'h')

for cell_id, grp in df.groupby('grid_cell', sort=False):
    grp = grp.sort_values('time_utc')
    times = grp['time_utc'].values
    mags  = grp['mag'].values
    idxs  = grp.index.values

    for pos in range(len(times)):
        t   = times[pos]
        i   = idxs[pos]
        dts = times[:pos] - t

        mask_24h = dts >= -TD_24H
        mask_7d  = dts >= -TD_7D

        past_mag_24h = mags[:pos][mask_24h]
        past_mag_7d  = mags[:pos][mask_7d]

        local_cnt_24h[i]    = len(past_mag_24h)
        local_max_24h[i]    = past_mag_24h.max() if len(past_mag_24h) else np.nan
        local_cnt_7d[i]     = len(past_mag_7d)
        if pos > 0:
            local_time_since[i] = (t - times[pos-1]) / np.timedelta64(1, 'h')

df['local_cnt_24h']     = local_cnt_24h
df['local_max_mag_24h'] = local_max_24h
df['local_cnt_7d']      = local_cnt_7d
df['local_time_since_h']= local_time_since

print('Пространственные признаки добавлены')
df[['time_utc','mag','grid_cell','local_cnt_24h','local_max_mag_24h','local_time_since_h']].head(8)

In [ ]:
features = [
    ('global_cnt_1h',       'Событий глобально\n(последний 1 час)'),
    ('global_cnt_24h',      'Событий глобально\n(последние 24 ч)'),
    ('global_max_mag_24h',  'Макс. магнитуда глобально\n(последние 24 ч)'),
    ('local_cnt_24h',       'Событий в ячейке  200 км\n(последние 24 ч)'),
    ('local_max_mag_24h',   'Макс. магнитуда в ячейке\n(последние 24 ч)'),
    ('local_time_since_h',  'Часов с последнего события\nв ячейке'),
]
colors = ['#4A90D9', '#E74C3C']
labels = ['Слабое (< 4.5)', 'Сильное (>= 4.5)']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, (feat, title) in zip(axes.flatten(), features):
    data = [df[df.is_strong == cls][feat].dropna() for cls in [0, 1]]
    parts = ax.violinplot(data, positions=[0, 1], showmedians=True, showextrema=False)
    for pc, color in zip(parts['bodies'], colors):
        pc.set_facecolor(color)
        pc.set_alpha(0.75)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(2.5)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold')

plt.suptitle('Новые признаки: распределение по классам', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
new_feats = [
    'global_cnt_1h','global_cnt_6h','global_cnt_24h','global_cnt_7d',
    'global_max_mag_24h','global_mean_mag_24h',
    'local_cnt_24h','local_cnt_7d',
    'local_max_mag_24h','local_time_since_h',
    'depth_km','latitude','longitude','hour',
]

corr_target = (
    df[new_feats + ['is_strong']]
    .corr()['is_strong']
    .drop('is_strong')
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = ['#E74C3C' if v > 0 else '#4A90D9' for v in corr_target.values]
bars = ax.barh(corr_target.index[::-1], corr_target.values[::-1],
               color=colors_bar[::-1], edgecolor='white', height=0.65)
ax.axvline(0, color='black', linewidth=1)
for bar, val in zip(bars, corr_target.values[::-1]):
    ax.text(val + (0.002 if val >= 0 else -0.002),
            bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=9)
ax.set_xlabel('Корреляция Пирсона с is_strong', fontsize=11)
ax.set_title('Корреляция признаков с целевой переменной', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Топ-5 признаков:')
print(corr_target.head(5).to_string())

In [ ]:
top_event = df.nlargest(1, 'mag').iloc[0]
t0   = top_event['time_utc']
cell = top_event['grid_cell']

window_days = 10
mask = (
    (df['grid_cell'] == cell) &
    (df['time_utc'] >= t0 - pd.Timedelta(days=window_days)) &
    (df['time_utc'] <= t0 + pd.Timedelta(days=window_days))
)
nearby = df[mask].copy()
nearby['hours_from_main'] = (nearby['time_utc'] - t0).dt.total_seconds() / 3600

fig, ax = plt.subplots(figsize=(13, 5))
ax.scatter(nearby[nearby['mag'] < 4.5]['hours_from_main'],
           nearby[nearby['mag'] < 4.5]['mag'],
           alpha=0.5, s=20, color='#4A90D9', label='Слабое', zorder=3)
ax.scatter(nearby[nearby['mag'] >= 4.5]['hours_from_main'],
           nearby[nearby['mag'] >= 4.5]['mag'],
           alpha=0.9, s=60, color='#E74C3C', label='Сильное', zorder=5)
ax.axvline(0, color='black', linewidth=2.5, linestyle='--', label='Главный толчок')
ax.axhline(4.5, color='grey', linewidth=1.2, linestyle=':', alpha=0.7)
ax.set_xlabel('Часов от главного толчка', fontsize=11)
ax.set_ylabel('Магнитуда', fontsize=11)
ax.set_title(
    f'Афтершоковая серия: {top_event["place"]} (mag {top_event["mag"]:.1f})\n'
    f'± {window_days} дней, ячейка  200×200 км',
    fontsize=12, fontweight='bold'
)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'Главный толчок: {top_event["place"]}')
print(f'Магнитуда: {top_event["mag"]}, Время: {t0}')
print(f'Событий в окне ±{window_days}d: {len(nearby)}')

**Выводы по новым признакам:**

| Признак | Вывод |
|---|---|
| `local_max_mag_24h` | Сильнейший предиктор: в активных афтершоковых сериях каждое последующее событие «видит» сильный предшественник -> высокая вероятность ещё одного сильного. Корреляция отражает реальную сейсмологию (закон Омори) |
| `local_cnt_24h` / `local_cnt_7d` | Умеренная положительная корреляция: чем активнее ячейка в прошлом, тем выше вероятность сильного события |
| `global_cnt_24h` | Слабее локальных - глобальный фон маскирует региональный сигнал |
| `local_time_since_h` | Отрицательная корреляция: короткий интервал с прошлым событием характерен для активных серий, где сильные события более вероятны. **Важно:** для первого события в ячейке признак = NaN ( 30–40% строк) - при обучении обязательна импутация (медиана или отдельный бинарный флаг) |
| Локальные vs глобальные | Локальные признаки информативнее - сейсмическая активность пространственно локализована в зонах разломов |

> **Критически важно для ML:** train/test split только по времени (`TimeSeriesSplit`).
> Случайное разбиение -> утечка данных из будущего -> завышенные метрики на тесте.

<a id='10b'></a>
## 10б. Форшоки: предшествующая активность перед сильными событиями

**Вопрос:** есть ли в данных предшествующая активность перед сильными (M ≥ 4.5) событиями?
Это проверяет, несёт ли `local_cnt_24h` реальный сигнал или просто шум.

> **Определение:** «форшок» здесь - операциональное: любое событие в той же ~200 км ячейке за 24 ч до сильного.
> Строгое сейсмологическое определение требует подтверждения принадлежности к одной последовательности.

In [ ]:
strong_events = df[df.is_strong == 1].copy()
strong_events['foreshocks_24h'] = strong_events['local_cnt_24h']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

clip_val = int(strong_events['foreshocks_24h'].quantile(0.97))
bins = np.arange(0, clip_val + 2) - 0.5
axes[0].hist(strong_events['foreshocks_24h'].clip(upper=clip_val),
             bins=bins, color='#E74C3C', alpha=0.85, edgecolor='white')
mean_fore = strong_events['foreshocks_24h'].mean()
axes[0].axvline(mean_fore, color='#F39C12', linewidth=2, linestyle='--',
                label=f'Среднее = {mean_fore:.1f}')
axes[0].set_xlabel('Событий в ячейке за 24 ч до сильного', fontsize=11)
axes[0].set_ylabel('Число сильных землетрясений', fontsize=11)
axes[0].set_title('Распределение числа форшоков', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)

sc = axes[1].scatter(strong_events['foreshocks_24h'].clip(upper=clip_val),
                     strong_events['mag'],
                     alpha=0.4, s=18,
                     c=strong_events['depth_km'],
                     cmap='plasma_r', vmin=0, vmax=300)
plt.colorbar(sc, ax=axes[1], label='Глубина (км)')
axes[1].set_xlabel('Число форшоков (24 ч, ~200 км)', fontsize=11)
axes[1].set_ylabel('Магнитуда сильного события', fontsize=11)
axes[1].set_title('Магнитуда vs число форшоков', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Статистика
has_foreshocks = (strong_events['foreshocks_24h'] > 0).sum()
total_strong = len(strong_events)
print(f'Сильных событий с форшоками: {has_foreshocks}/{total_strong} ({has_foreshocks/total_strong*100:.1f}%)')
print(f'Среднее число форшоков: {strong_events["foreshocks_24h"].mean():.2f}')
print(f'Макс. форшоков перед одним событием: {strong_events["foreshocks_24h"].max()}')

**Выводы по форшокам:**

| Наблюдение | Интерпретация |
|---|---|
| Большинство сильных событий имеют 0 форшоков | Подтверждает, что предсказание единичных «внезапных» событий остаётся крайне сложным |
| Часть сильных событий (28%) имеют предшественников | Это вторичные толчки в афтершоковых сериях - они «видят» предыдущую активность |
| Нет выраженной связи числа форшоков с магнитудой | Более сильное событие не обязательно предваряется большим числом форшоков |

**Сейсмологический контекст:**
- Большинство сильных землетрясений случаются без предупреждения - в этом и состоит сложность задачи
- Тем не менее **совместный сигнал** `local_cnt_24h` + `local_max_mag_24h` статистически значим:
  активная серия в ячейке увеличивает вероятность следующего сильного события
- Признак `local_cnt_24h` несёт **реальную информацию**, а не случайный шум

> Для прогноза важен не изолированный форшок, а **паттерн нарастающей активности**:
> рост `local_cnt_24h` и высокое `local_max_mag_24h` вместе - наиболее информативная комбинация.

<a id='11'></a>
## 11. Итоговый отчёт

In [ ]:
summary = {
    'Всего событий': len(df),
    'Период': f"{df.time_utc.min().date()} - {df.time_utc.max().date()}",
    'Дней охвата': (df.time_utc.max() - df.time_utc.min()).days,
    'Сильных (mag >= 4.5)': df['is_strong'].sum(),
    'Доля сильных': f"{df['is_strong'].mean()*100:.2f}%",
    'Соотношение классов': f"{(1-df['is_strong'].mean())/df['is_strong'].mean():.1f}:1",
    'Медиана магнитуды': df['mag'].median(),
    'Макс. магнитуда': df['mag'].max(),
    'Медиана глубины (км)': df['depth_km'].median(),
    'Записей с отриц. магнитудой': (df['mag'] < 0).sum(),
    'Записей с отриц. глубиной': (df['depth_km'] < 0).sum(),
    'Пропущенных значений': df.isnull().sum().sum(),
    'Самый активный регион': df['region'].value_counts().index[0],
}

print('         СВОДНЫЙ ОТЧЁТ ПО ДАТАСЕТУ')
for k, v in summary.items():
    print(f'  {k:<35} {v}')

---

## Выводы

### Ключевые находки

| # | Вывод |
|---|-------|
| 1 | **Дисбаланс классов 19:1** - сильные землетрясения составляют лишь 5% событий. Это ключевая проблема при обучении. |
| 2 | **Закон Гутенберга–Рихтера подтверждён:** число событий экспоненциально убывает с ростом магнитуды. b = 1.0 глобально, но варьируется по регионам. |
| 3 | **Отрицательные магнитуды** - норма для современных сейсмографов, не удалять. **Отрицательные глубины** - артефакт алгоритма локации USGS («близко к поверхности»), клипировать до 0 при необходимости. |
| 4 | **Глубина слабо коррелирует с магнитудой** (r = 0.15). Глубокофокусные события (> 300 км) в датасете редки (1%) - их статистика нестабильна, делать выводы об «опасности» некорректно. |
| 5 | **Большинство событий мелкофокусные** (глубина 0–70 км) - 88% датасета. Глубина > 70 км - косвенный маркёр субдукционной обстановки. |
| 6 | **b-value различается по регионам:** зоны субдукции (b < 0.9) несут повышенную долю сильных событий; вулканические/дисперсные зоны (b > 1.1) - меньше. Это потенциально полезный региональный признак. |
| 7 | **Большинство сильных событий происходит без форшоков** в той же ячейке за 24 ч. Тем не менее совместный сигнал `local_cnt_24h` + `local_max_mag_24h` статистически значим и отражает афтершоковые серии. |
| 8 | **Пространственно-временные признаки** (`local_max_mag_24h`, `local_cnt_24h`) - наиболее информативные предикторы; отражают закон Омори и кластеризацию афтершоков. |
| 9 | **Зона Вадати–Бениоффа** видна на региональных срезах. Сильные события (M >= 4.5) концентрируются на глубинах до 150 км, где напряжения в погружающейся плите максимальны. |
| 10 | **Временные признаки (час UTC, день недели)** не несут прогностической ценности - данные не показывают значимой зависимости, что ожидаемо - физически землетрясения не зависят от времени суток на Земле. |

### ML-этап

**Признаки для модели (в порядке значимости по EDA):**
- Пространственно-временные: `local_max_mag_24h`, `local_cnt_24h`, `local_cnt_7d`, `local_time_since_h` (импутация NaN), `global_cnt_24h`
- Пространственные: `latitude`, `longitude`, `depth_km`, `abs_latitude`, `depth_cat_num`
- Региональные: b-value на уровне пространственной ячейки (при достаточной истории)
- Слабые, но проверить: `hour` (UTC), `depth_x_abs_lat`

**Предобработка:**
- Клипировать `depth_km` до 0 снизу (отрицательные = поверхностные события)
- Заполнить NaN в `local_time_since_h` (медиана или бинарный флаг `is_first_in_cell`)
- Train/test split **только по времени** (`TimeSeriesSplit` или ручной порог по дате)

**Борьба с дисбалансом:**
- `class_weight='balanced'` в алгоритме
- SMOTE / ADASYN для оверсэмплинга (только на train!)
- Метрики: **F1, AUC-ROC** - не accuracy!

**Алгоритмы-кандидаты:**
- Gradient Boosting (XGBoost, LightGBM) - хорошо работают с табличными данными и дисбалансом
- Random Forest - как ансамблевый baseline
- Logistic Regression - как линейный baseline для сравнения

Главный прогностический сигнал - не отдельный признак,
а КОМБИНАЦИЯ: пространственная локализация (grid_cell) +
история активности (local_max_mag_24h) + тектонический
контекст (b-value региона + depth_cat).

---
*Анализ выполнен на данных USGS Earthquake API, 180 дней, 20 000 событий.*

<a id='12'></a>
## 12. Baseline-модель

**Задача:** бинарная классификация - предсказать `is_strong = (mag ≥ 4.5)` на основе
сейсмического контекста, координат и глубины, **без использования самой магнитуды
текущего события** (иначе утечка таргета).

**Ключевые решения:**

| Проблема | Решение |
|---|---|
| Дисбаланс классов 19:1 | `class_weight='balanced'`, метрики PR-AUC / F1 вместо accuracy |
| Временной ряд | Сплит по времени (последние 20% - test); никакого `shuffle` |
| NaN в `local_*` для первого события в ячейке | Импутация: `0` для max-mag, `168 ч` (7 дней) для time_since |
| Отрицательные глубины (артефакт USGS) | `clip(lower=0)` |

**Состав сравнения:**
- `Dummy` (most_frequent / stratified) - нижняя граница «без модели»
- `Logistic Regression` - линейный baseline со скейлером
- `Random Forest` - нелинейный, интерпретируемый по важности признаков
- `HistGradientBoosting` - современный бустинг, нативно работает с NaN


### 12.1. Подготовка признаков и сплит

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve,
)

FEATURES = [
    # Координаты и глубина
    'depth_km', 'latitude', 'longitude', 'abs_latitude',
    # Временные
    'hour', 'dayofweek', 'month',
    # Инженерные (координаты × глубина)
    'lat_zone_num', 'depth_cat_num', 'depth_x_abs_lat', 'depth_lat_interact',
    # Глобальный сейсмический контекст
    'global_cnt_1h', 'global_cnt_6h', 'global_cnt_24h', 'global_cnt_7d',
    'global_max_mag_24h', 'global_mean_mag_24h',
    # Локальный (по 2°×2° ячейкам)
    'local_cnt_24h', 'local_cnt_7d',
    'local_max_mag_24h', 'local_time_since_h',
]
TARGET = 'is_strong'

data = df.sort_values('time_utc').reset_index(drop=True).copy()

data['local_max_mag_24h']   = data['local_max_mag_24h'].fillna(0.0)
data['local_time_since_h']  = data['local_time_since_h'].fillna(168.0)
data['global_max_mag_24h']  = data['global_max_mag_24h'].fillna(0.0)
data['global_mean_mag_24h'] = data['global_mean_mag_24h'].fillna(0.0)
data['depth_km']            = data['depth_km'].clip(lower=0)

data[FEATURES] = data[FEATURES].fillna(0.0)

X = data[FEATURES].values
y = data[TARGET].values

split_idx  = int(len(data) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
split_time = data['time_utc'].iloc[split_idx]

print(f'Признаков: {len(FEATURES)}')
print(f'Train: {X_train.shape}, доля позитивов: {y_train.mean():.3%}')
print(f'Test:  {X_test.shape}, доля позитивов: {y_test.mean():.3%}')
print(f'Граница сплита по времени: {split_time}')


### 12.2. Обучение моделей и сравнение метрик


In [ ]:
models = {
    'Dummy (most_frequent)': DummyClassifier(strategy='most_frequent'),
    'Dummy (stratified)':    DummyClassifier(strategy='stratified', random_state=42),
    'Logistic Regression':   Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000,
                                   class_weight='balanced',
                                   random_state=42)),
    ]),
    'Random Forest':         RandomForestClassifier(
        n_estimators=300, max_depth=12,
        class_weight='balanced',
        n_jobs=-1, random_state=42,
    ),
    'HistGradientBoosting':  HistGradientBoostingClassifier(
        max_iter=400, max_depth=8, learning_rate=0.05,
        class_weight='balanced', random_state=42,
    ),
}

results = []
preds = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_test)[:, 1]
    else:
        proba = model.predict(X_test).astype(float)
    pred = (proba >= 0.5).astype(int)
    preds[name] = (proba, pred)

    roc_auc = roc_auc_score(y_test, proba) if len(np.unique(proba)) > 1 else 0.5
    results.append({
        'Модель':    name,
        'ROC-AUC':   roc_auc,
        'PR-AUC':    average_precision_score(y_test, proba),
        'F1':        f1_score(y_test, pred, zero_division=0),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall':    recall_score(y_test, pred, zero_division=0),
        'Accuracy':  accuracy_score(y_test, pred),
    })

results_df = (pd.DataFrame(results)
              .set_index('Модель')
              .round(4)
              .sort_values('PR-AUC', ascending=False))
results_df.style.background_gradient(cmap='YlGn', subset=['ROC-AUC','PR-AUC','F1'])


### 12.3. ROC и Precision–Recall кривые


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

for name, (proba, _) in preds.items():
    if name.startswith('Dummy'):
        continue
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

axes[0].plot([0, 1], [0, 1], '--', color='gray', alpha=0.5, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC-кривые', fontsize=12, fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

baseline_p = y_test.mean()
for name, (proba, _) in preds.items():
    if name.startswith('Dummy'):
        continue
    p, r, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[1].plot(r, p, label=f'{name} (AP={ap:.3f})', linewidth=2)

axes[1].axhline(baseline_p, linestyle='--', color='gray', alpha=0.5,
                label=f'Random baseline (P={baseline_p:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision–Recall кривые', fontsize=12, fontweight='bold')
axes[1].legend(loc='upper right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


### 12.4. Confusion matrix лучшей модели


In [ ]:
best_name = results_df.index[0]
proba_best, pred_best = preds[best_name]

cm = confusion_matrix(y_test, pred_best)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Слабое', 'Сильное'],
            yticklabels=['Слабое', 'Сильное'], ax=axes[0])
axes[0].set_xlabel('Предсказано')
axes[0].set_ylabel('Истинно')
axes[0].set_title(f'Counts - {best_name}', fontweight='bold')

sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', cbar=False,
            xticklabels=['Слабое', 'Сильное'],
            yticklabels=['Слабое', 'Сильное'], ax=axes[1])
axes[1].set_xlabel('Предсказано')
axes[1].set_ylabel('Истинно')
axes[1].set_title('Normalized по строке (recall per class)', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\nClassification report - {best_name}:\n')
print(classification_report(
    y_test, pred_best,
    target_names=['Слабое (mag<4.5)', 'Сильное (mag>=4.5)'],
    digits=3,
))


### 12.5. Важность признаков (Random Forest)


In [ ]:
rf = models['Random Forest']
imp = (pd.DataFrame({'feature': FEATURES, 'importance': rf.feature_importances_})
       .sort_values('importance', ascending=True))

fig, ax = plt.subplots(figsize=(10, 7))
colors_imp = sns.color_palette('viridis', len(imp))
ax.barh(imp['feature'], imp['importance'], color=colors_imp, edgecolor='white')
for i, (feat, val) in enumerate(zip(imp['feature'], imp['importance'])):
    ax.text(val + 0.001, i, f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('Важность (Gini-importance)')
ax.set_title('Важность признаков, Random Forest', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Топ-7 признаков:')
print(imp.tail(7).iloc[::-1].to_string(index=False))


### 12.6. Сводная диаграмма метрик


In [ ]:
metrics_to_plot = ['ROC-AUC', 'PR-AUC', 'F1', 'Recall']
plot_df = results_df[metrics_to_plot]

fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(plot_df))
width = 0.2
colors_m = ['#4A90D9', '#E74C3C', '#27AE60', '#F39C12']

for i, metric in enumerate(metrics_to_plot):
    bars = ax.bar(x + (i - 1.5) * width, plot_df[metric].values,
                  width, label=metric, color=colors_m[i], edgecolor='white')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.2f}', ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(plot_df.index, rotation=15, ha='right')
ax.set_ylabel('Значение метрики')
ax.set_ylim(0, 1.05)
ax.set_title('Сравнение моделей на тесте', fontsize=12, fontweight='bold')
ax.legend(loc='upper right', ncol=4)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


### 12.7. Выводы baseline

| # | Наблюдение |
|---|-----------|
| 1 | Все ML-модели существенно превосходят dummy по PR-AUC - в данных есть предиктивный сигнал, который не сводится к выбору большинства класса. |
| 2 | **Главные предикторы - признаки сейсмического контекста** (`local_max_mag_24h`, `global_max_mag_24h`, `global_cnt_24h`). Это согласуется с физикой: сильные события группируются в активных сериях (афтершоки + закон Омори). |
| 3 | Календарные признаки (`hour`, `dayofweek`, `month`) - почти нулевая важность. Землетрясения не имеют суточной/недельной сезонности - подтверждение наблюдения из EDA. |
| 4 | На пороге 0.5 модели дают высокий recall, но низкий precision: следствие дисбаланса 19:1. На практике порог нужно подбирать под метрику (cost-sensitive). |
| 5 | HistGradientBoosting и Random Forest близки по PR-AUC. Бустинг привлекательнее в production: быстрее на инференсе и нативно работает с NaN. |

**Что НЕ вошло в baseline осознанно:**
- `mag` среди фич - иначе утечка таргета (целевая переменная - её бинаризация).
- `region` как one-hot - высокая кардинальность; для следующей итерации лучше target-encoding с time-aware кросс-валидацией.
- Калибровка вероятностей (`CalibratedClassifierCV`) - нужна, если использовать `proba` напрямую как риск.

**Куда улучшать:**
1. **TimeSeriesSplit + подбор гиперпараметров** (Optuna) - сейчас одна точка в пространстве моделей.
2. **Подбор порога** по F-beta или по cost-matrix (FN сильнее FP для сейсмической задачи).
3. **Дополнительные признаки:** b-value по локальной ячейке (есть в EDA для регионов), скользящие квантили глубины, расстояние до ближайшего исторического сильного события.
4. **Downsampling класса 0** или `scale_pos_weight` в XGBoost/LightGBM - на полных данных USGS возможен значимый buster по PR-AUC.


<a id='13'></a>
### 13. Итог по аномалиям и выбросам

В задании требовалось «разобраться, где аномалии и выбросы мешают, а где, наоборот, могут дать важные инсайты», но мы уже это сделали на этапе EDA. Ниже сводный ответ - со ссылками на ячейки, где конкретные проверки уже проведены.

#### Что обнаружено и как обработано

| Тип аномалии | Где найдено | Природа | Решение |
|---|---|---|---|
| Пропуски в основных колонках | секция 2, cell с `df.isnull().sum()` | Отсутствуют (датасет USGS чистый) | Ничего не делаем |
| Дубликаты | секция 2 | Отсутствуют (`id` уникален) | Ничего не делаем |
| **Отрицательная глубина** | секция 2 (cell с `neg_depth`) | Артефакт алгоритма гипоцентральной локации USGS - означает «приповерхностное» событие | **Сохраняем** как отдельную категорию `Аномальная (< 0)`; в модели - `clip(lower=0)` |
| **Магнитуда <= 0** | секция 2 | Корректные физические значения (логарифмическая шкала) | Сохраняем |
| Сверхдальние глубины (>400 км) | секция 5, cell с `hist2d` | Реальные глубокофокусные события из субдукционных зон | Сохраняем; **это инсайт**, а не выброс |
| Высокие магнитуды (mag >= 4.5) | секция 3, target | **Это и есть таргет**, по определению хвост распределения | Сохраняем - модель учится именно на них |
| Длинный хвост `local_cnt_24h` (рои/афтершоки) | секция 10б | Афтершоковые серии - закон Омори | Сохраняем; клиппинг `quantile(0.97)` только для визуализации |
| NaN в `local_max_mag_24h`, `local_time_since_h` | секция 10 | «Холодный старт» - первое событие в ячейке | Импутация (0 и 168 ч) в baseline |

#### Где аномалии **мешают**

1. **Отрицательные глубины в модели**: масштабируют скейлеры (LogReg) и портят корреляции → `clip(lower=0)` перед обучением.
2. **NaN в скользящих окнах** на старте периода: ломают `LogReg` / `RF` → импутация перед baseline.
3. **Визуализация**: длинные хвосты сжимают основную массу - кропы (`depth ≤ 400`, `quantile(0.97)`) нужны только для графиков, не для модели.

#### Где аномалии - **источник инсайтов**

1. **Глубокофокусные события (>300 км)** локализованы в зоне Вадати–Бениоффа → прямое свидетельство субдукции (секция 9б).
2. **Высокий `local_cnt_24h`** перед сильным толчком = форшоковая активность (секция 10б) - оказался в топ-3 по важности признаков в baseline (секция 12.5).
3. **Низкий b-value по региону** (наклон Гутенберга–Рихтера, секция 9в) выделяет «запертые» разломы - регионы с повышенной долей сильных событий.
4. **Отсутствие сезонности** (`hour`, `month` ≈ 0 по важности в RF) - это тоже «аномалия» относительно бытовой интуиции о землетрясениях, и она важна: подтверждает физическую независимость событий от календаря.

#### Принятый принцип

> **Не удаляем ни одного события.** В сейсмологии «выбросы» по статистическим критериям (IQR, Z-score) почти всегда являются полезным сигналом - экстремум распределения = редкое сильное событие, которое мы и хотим предсказывать. Вмешательство ограничивается:
> - `clip(lower=0)` для отрицательной глубины (артефакт измерения),
> - `fillna` для NaN на стартовых границах rolling-окон,
> - визуальными кропами на графиках без изменения данных.


# Этап 2. Работа с аномалиями и генерация признаков

## 14. Углублённый анализ выбросов

На Этапе 1 (секции 2, 5, 13) мы уже сделали **обзорную** проверку аномалий — посмотрели на пропуски, дубликаты, отрицательные глубины и хвосты распределений по бизнес-логике сейсмологии. Этот блок — **углублённый формальный анализ**, который требуется заданием Этапа 2: три статистических метода (Z-оценка, IQR, тест Граббса) и ML-методы (Isolation Forest, LOF) с количественными метриками качества.

Что нового по сравнению с Этапом 1:

- применяем **формальные критерии** выявления выбросов, а не только визуальный осмотр;
- строим **флаги-признаки** на основе результатов методов и добавляем их в датасет;
- оцениваем ML-методы по **Precision / Recall / F1 / ROC-AUC** относительно `is_strong`;
- сравниваем методы между собой: какие точки они помечают согласованно, а где расходятся.


### 14.1. Z-оценка


In [ ]:
from scipy import stats

num_cols_for_outliers = ['mag', 'depth_km', 'latitude', 'longitude']
z_scores = pd.DataFrame()
for col in num_cols_for_outliers:
    z_scores[col] = np.abs(stats.zscore(df[col].fillna(df[col].median())))

z_outliers = (z_scores > 3).sum()
print('Число выбросов по Z-score > 3:')
print(z_outliers)
print(f'\nВсего строк: {len(df)}')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, num_cols_for_outliers):
    ax.hist(z_scores[col], bins=60, color='#4A90D9', edgecolor='white')
    ax.axvline(3, color='#E74C3C', linestyle='--', linewidth=2, label='Z=3')
    ax.set_title(col)
    ax.set_xlabel('|Z|')
    ax.legend()
plt.tight_layout()
plt.show()

Z-score хорошо ловит выбросы по магнитуде - это как раз сильные землетрясения, которые мы и пытаемся предсказывать. По глубине Z-score также находит реальные глубокофокусные события (500+ км). То есть формальные "выбросы" - на самом деле полезный сигнал.


### 14.2. IQR (межквартильный размах)


In [ ]:
def iqr_outliers(series, k=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - k * iqr, q3 + k * iqr
    return (series < low) | (series > high), (low, high)

iqr_flags = {}
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, num_cols_for_outliers):
    flag, (low, high) = iqr_outliers(df[col])
    iqr_flags[col] = flag
    ax.boxplot(df[col].dropna(), vert=True)
    ax.set_title(f'{col}\nвыбросов: {flag.sum()} ({flag.mean()*100:.1f}%)')
    ax.set_ylabel(col)
plt.tight_layout()
plt.show()

iqr_df = pd.DataFrame({col: [iqr_flags[col].sum()] for col in num_cols_for_outliers},
                     index=['IQR выбросов'])
print(iqr_df.T)

IQR более агрессивный метод - помечает как выбросы много точек. По магнитуде помечает всё что выше ~3.0, что включает почти все сильные землетрясения. Видно что IQR в нашей задаче не подходит для "очистки": слишком много полезного сигнала будет потеряно.


### 14.3. Тест Граббса


In [ ]:
def grubbs_test(data, alpha=0.05):
    data = np.array(data)
    n = len(data)
    if n < 3:
        return None
    mean_ = data.mean()
    std_ = data.std(ddof=1)
    if std_ == 0:
        return None
    g_calc = np.max(np.abs(data - mean_)) / std_
    t_crit = stats.t.ppf(1 - alpha / (2 * n), n - 2)
    g_crit = (n - 1) / np.sqrt(n) * np.sqrt(t_crit**2 / (n - 2 + t_crit**2))
    idx = np.argmax(np.abs(data - mean_))
    return {'G_calc': g_calc, 'G_crit': g_crit,
            'is_outlier': g_calc > g_crit,
            'value': data[idx], 'index': idx}

print('Тест Граббса (проверяем самое экстремальное значение)')
for col in num_cols_for_outliers:
    res = grubbs_test(df[col].dropna().values)
    if res is not None:
        flag = 'ДА' if res['is_outlier'] else 'нет'
        print(f"{col:>12}: G={res['G_calc']:.2f}, G_crit={res['G_crit']:.2f}, "
              f"выброс: {flag}, значение={res['value']:.2f}")

Тест Граббса подтверждает: максимальные значения магнитуды и глубины статистически значимо отличаются от основной массы. Но это не повод их удалять - это редкие, но физически реальные события. Решение: оставляем как есть, но создадим бинарные флаги выбросов как признаки.


### 14.4. Признаки-флаги на основе выбросов


In [ ]:
for col in ['mag', 'depth_km']:
    z = np.abs(stats.zscore(df[col].fillna(df[col].median())))
    df[f'{col}_z_outlier'] = (z > 3).astype(int)

q1, q3 = df['depth_km'].quantile([0.25, 0.75])
iqr = q3 - q1
df['depth_iqr_outlier'] = ((df['depth_km'] < q1 - 1.5*iqr) |
                            (df['depth_km'] > q3 + 1.5*iqr)).astype(int)

df['has_any_outlier'] = (
    df['mag_z_outlier'] + df['depth_km_z_outlier'] + df['depth_iqr_outlier']
).clip(upper=1)

print('Доля флагов выбросов по классам (mean):')
print(df.groupby('is_strong')[
    ['mag_z_outlier', 'depth_km_z_outlier', 'depth_iqr_outlier', 'has_any_outlier']
].mean().round(3))

Видно: у сильных событий доля флагов выбросов выше. Это значит признаки будут полезными для модели - они отделяют "хвост" распределения, где живут сильные землетрясения.


## 15. ML-методы поиска выбросов

Применим Isolation Forest и Local Outlier Factor (LOF) - два популярных метода поиска аномалий. Сравним с целевой переменной `is_strong` как с псевдо-меткой "интересных" точек.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

ml_features = ['mag', 'depth_km', 'latitude', 'longitude',
               'global_cnt_24h', 'local_cnt_24h']
X_anom = df[ml_features].fillna(df[ml_features].median())
scaler = StandardScaler()
X_anom_sc = scaler.fit_transform(X_anom)

contamination = df['is_strong'].mean()
print(f'Доля положительного класса: {contamination*100:.2f}%')

iso = IsolationForest(contamination=contamination, random_state=42, n_estimators=200)
df['iso_score'] = -iso.fit(X_anom_sc).score_samples(X_anom_sc)
df['iso_flag'] = (iso.fit_predict(X_anom_sc) == -1).astype(int)

lof = LocalOutlierFactor(n_neighbors=30, contamination=contamination)
df['lof_flag'] = (lof.fit_predict(X_anom_sc) == -1).astype(int)
df['lof_score'] = -lof.negative_outlier_factor_

print(f"IsolationForest пометил аномалий: {df['iso_flag'].sum()}")
print(f"LOF пометил аномалий: {df['lof_flag'].sum()}")

In [ ]:
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report)

print('Качество детекции "сильных событий" как аномалий:\n')
for method, flag, score in [
    ('IsolationForest', df['iso_flag'], df['iso_score']),
    ('LOF',             df['lof_flag'], df['lof_score']),
]:
    p = precision_score(df['is_strong'], flag)
    r = recall_score(df['is_strong'], flag)
    f = f1_score(df['is_strong'], flag)
    a = roc_auc_score(df['is_strong'], score)
    print(f"{method:>18}: P={p:.3f}  R={r:.3f}  F1={f:.3f}  ROC-AUC={a:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, method, score_col in [
    (axes[0], 'IsolationForest', 'iso_score'),
    (axes[1], 'LOF',             'lof_score'),
]:
    ax.scatter(df.loc[df.is_strong == 0, 'depth_km'],
               df.loc[df.is_strong == 0, 'mag'],
               c=df.loc[df.is_strong == 0, score_col],
               cmap='Blues', s=6, alpha=0.4)
    ax.scatter(df.loc[df.is_strong == 1, 'depth_km'],
               df.loc[df.is_strong == 1, 'mag'],
               color='#E74C3C', s=18, alpha=0.85, label='Сильное событие')
    ax.set_xlabel('Глубина (км)')
    ax.set_ylabel('Магнитуда')
    ax.set_title(method)
    ax.legend()

plt.suptitle('ML-методы: точки и их аномальный скор', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Сравнение методов:**

- IsolationForest даёт более стабильный ROC-AUC, потому что строит ансамбль деревьев и устойчив к шуму.
- LOF чувствителен к локальной плотности - хорошо ловит точечные аномалии, но даёт больше шума на больших данных.
- Оба метода частично совпадают с целевой переменной: сильные события действительно находятся в "редкой" области пространства признаков. Но ни один метод не заменяет supervised-классификатор, потому что они не используют метки.

Сохраняем флаги `iso_flag`, `lof_flag` и сами скоры как признаки для последующей модели.


## 16. Обработка категориальных переменных

В данных есть категориальные признаки: `region` (высокая кардинальность) и `depth_cat` (4 значения). Применим разные методы кодирования.


### 16.1. One-Hot Encoding для depth_cat


In [ ]:
ohe_depth = pd.get_dummies(df['depth_cat'], prefix='depth', dtype=int)
df = pd.concat([df, ohe_depth], axis=1)
print('Колонки после OHE:', list(ohe_depth.columns))
ohe_depth.head()

### 16.2. Label Encoding для region


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['region_le'] = le.fit_transform(df['region'].fillna('UNKNOWN'))
print(f'Уникальных регионов: {df["region_le"].nunique()}')
print('Топ-5 регионов:')
print(df['region'].value_counts().head())

Label Encoding для `region` подходит как простой baseline. Но для линейных моделей это плохо: порядковая шкала никак не отражает близость регионов. Поэтому в следующем шаге применим Target Encoding.


### 16.3. Target Encoding для region (с защитой от утечки)


In [ ]:
# Делаем target encoding только на train (первые 80% по времени),
# а на test применяем уже посчитанные значения - так избегаем утечки.
df_sorted = df.sort_values('time_utc').reset_index(drop=True)
split_idx = int(len(df_sorted) * 0.8)

train_part = df_sorted.iloc[:split_idx]
test_part  = df_sorted.iloc[split_idx:]

global_mean = train_part['is_strong'].mean()
region_stats = train_part.groupby('region')['is_strong'].agg(['mean', 'count'])

# Сглаживание (smoothing) чтобы не переобучиться на маленьких регионах
m = 20  # параметр сглаживания
region_stats['te'] = ((region_stats['mean'] * region_stats['count'] + global_mean * m)
                      / (region_stats['count'] + m))

te_map = region_stats['te'].to_dict()

df_sorted['region_te'] = df_sorted['region'].map(te_map).fillna(global_mean)
df = df.merge(df_sorted[['id', 'region_te']], on='id', how='left')

print('Распределение region_te:')
print(df['region_te'].describe().round(4))
print(f'\nГлобальное среднее на train: {global_mean:.4f}')

**Выбор кодирования:**
- `depth_cat` - 4 значения, OHE подходит идеально.
- `region` - сотни значений. Label Encoding оставим как простой baseline, но в финальных моделях используем `region_te` - оно несёт информацию о "рискованности" региона и при этом устойчиво (за счёт сглаживания).
- Утечки нет: статистика `region_te` посчитана только по train, на test применяется уже зафиксированный словарь.


## 17. Признаки на основе ближайших соседей

У нас есть координаты, поэтому считаем kNN по координатам и собираем агрегаты по соседям. Используем BallTree с метрикой haversine - это правильное расстояние на сфере.


In [ ]:
from sklearn.neighbors import BallTree

coords_rad = np.radians(df[['latitude', 'longitude']].values)
tree = BallTree(coords_rad, metric='haversine')

K = 10
distances, indices = tree.query(coords_rad, k=K + 1)
# первый сосед - сама точка, отбрасываем
distances = distances[:, 1:]
indices = indices[:, 1:]

EARTH_R = 6371.0
df['knn_mean_dist_km']   = distances.mean(axis=1) * EARTH_R
df['knn_min_dist_km']    = distances.min(axis=1) * EARTH_R
df['knn_density']        = 1.0 / (df['knn_mean_dist_km'] + 1e-3)

mags = df['mag'].values
df['knn_mean_mag']  = mags[indices].mean(axis=1)
df['knn_max_mag']   = mags[indices].max(axis=1)

depth = df['depth_km'].values
df['knn_mean_depth'] = depth[indices].mean(axis=1)

print('Признаки соседей добавлены.')
df[['knn_mean_dist_km', 'knn_density', 'knn_mean_mag', 'knn_max_mag']].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
neighbor_feats = ['knn_mean_mag', 'knn_max_mag', 'knn_density']
for ax, feat in zip(axes, neighbor_feats):
    data_0 = df[df.is_strong == 0][feat]
    data_1 = df[df.is_strong == 1][feat]
    ax.hist(data_0, bins=40, alpha=0.5, color='#4A90D9', label='Слабое', density=True)
    ax.hist(data_1, bins=40, alpha=0.5, color='#E74C3C', label='Сильное', density=True)
    ax.set_title(feat)
    ax.legend()
plt.tight_layout()
plt.show()

print('Корреляция признаков-соседей с is_strong:')
print(df[neighbor_feats + ['knn_mean_dist_km', 'is_strong']].corr()['is_strong'].drop('is_strong'))

**Гипотезы по признакам соседей:**
- `knn_max_mag` - если рядом уже было сильное событие, повышается вероятность что и эта точка - сильная (афтершоковая серия).
- `knn_density` - в активных сейсмических зонах точки расположены плотнее, чем в пассивных. Высокая плотность = "горячая" зона.
- `knn_mean_dist_km` - обратное к density.

Распределения для слабых и сильных событий различаются - значит признаки несут полезный сигнал.


## 18. Циклическое кодирование времени

Час, день недели и месяц - циклические величины. Для линейных моделей кодирование числами 0-23 создаёт ложную "большую" разницу между 23 и 0 (на самом деле они рядом). Поэтому используем sin/cos.


In [ ]:
df['hour_sin']  = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos']  = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin']   = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos']   = np.cos(2 * np.pi * df['dayofweek'] / 7)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(df['hour_sin'], df['hour_cos'], c=df['hour'], cmap='hsv', s=4, alpha=0.5)
ax.set_xlabel('hour_sin')
ax.set_ylabel('hour_cos')
ax.set_title('Циклическое кодирование часа суток')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

Точки лежат на окружности - именно то что нужно. По EDA мы уже знаем, что временные признаки имеют слабую связь с целью, но добавляем их корректно закодированными - вдруг в комбинации с другими признаками появится сигнал.


## 19. Отбор признаков

Применяем три подхода: фильтры (корреляции, ANOVA), обёртки (RFECV) и встроенные методы (L1, важности из бустинга).


In [ ]:
ALL_FEATURES = [
    'depth_km', 'latitude', 'longitude', 'abs_latitude',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'lat_zone_num', 'depth_cat_num', 'depth_x_abs_lat', 'depth_lat_interact',
    'global_cnt_1h', 'global_cnt_6h', 'global_cnt_24h', 'global_cnt_7d',
    'global_max_mag_24h', 'global_mean_mag_24h',
    'local_cnt_24h', 'local_cnt_7d',
    'local_max_mag_24h', 'local_time_since_h',
    'mag_z_outlier', 'depth_km_z_outlier', 'depth_iqr_outlier',
    'iso_score', 'iso_flag', 'lof_score', 'lof_flag',
    'region_le', 'region_te',
    'knn_mean_dist_km', 'knn_density', 'knn_mean_mag', 'knn_max_mag', 'knn_mean_depth',
]
ohe_cols = [c for c in df.columns if c.startswith('depth_') and c not in
            ['depth_km', 'depth_cat', 'depth_cat_num', 'depth_x_abs_lat',
             'depth_lat_interact', 'depth_km_z_outlier', 'depth_iqr_outlier']]
ALL_FEATURES += ohe_cols

# Защита от дублей (на случай если что-то добавится повторно)
ALL_FEATURES = list(dict.fromkeys(ALL_FEATURES))

print(f'Всего признаков-кандидатов: {len(ALL_FEATURES)}')

data_full = df.sort_values('time_utc').reset_index(drop=True).copy()
data_full['local_max_mag_24h']   = data_full['local_max_mag_24h'].fillna(0.0)
data_full['local_time_since_h']  = data_full['local_time_since_h'].fillna(168.0)
data_full['global_max_mag_24h']  = data_full['global_max_mag_24h'].fillna(0.0)
data_full['global_mean_mag_24h'] = data_full['global_mean_mag_24h'].fillna(0.0)
data_full['depth_km']            = data_full['depth_km'].clip(lower=0)
data_full[ALL_FEATURES] = data_full[ALL_FEATURES].fillna(0.0)

split_idx2 = int(len(data_full) * 0.8)
X_all = data_full[ALL_FEATURES].values
y_all = data_full['is_strong'].values

X_tr, X_te = X_all[:split_idx2], X_all[split_idx2:]
y_tr, y_te = y_all[:split_idx2], y_all[split_idx2:]
print(f'Train: {X_tr.shape}, Test: {X_te.shape}')


### 19.1. Фильтры: корреляции и ANOVA F-test


In [ ]:
from sklearn.feature_selection import f_classif

corr_with_y = pd.Series(
    [np.corrcoef(X_tr[:, i], y_tr)[0, 1] for i in range(X_tr.shape[1])],
    index=ALL_FEATURES
).abs().sort_values(ascending=False)

f_vals, p_vals = f_classif(X_tr, y_tr)
anova = pd.DataFrame({'F': f_vals, 'p': p_vals}, index=ALL_FEATURES).sort_values('F', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
top_corr = corr_with_y.head(15)[::-1]
axes[0].barh(top_corr.index, top_corr.values, color='#4A90D9')
axes[0].set_title('Топ-15 по |корреляции| с is_strong')
axes[0].set_xlabel('|Pearson|')

top_anova = anova.head(15)[::-1]
axes[1].barh(top_anova.index, top_anova['F'].values, color='#E74C3C')
axes[1].set_title('Топ-15 по ANOVA F-статистике')
axes[1].set_xlabel('F')
plt.tight_layout()
plt.show()

print('Самые слабые признаки (низкий F):')
print(anova.tail(8))

### 19.2. Встроенные методы: L1-регуляризация


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)

l1_model = LogisticRegression(penalty='l1', solver='liblinear', C=0.1,
                              class_weight='balanced', random_state=42, max_iter=1000)
l1_model.fit(X_tr_sc, y_tr)

l1_coef = pd.Series(l1_model.coef_[0], index=ALL_FEATURES)
nonzero = l1_coef[l1_coef != 0].abs().sort_values(ascending=False)

print(f'Признаков с ненулевым коэффициентом: {(l1_coef != 0).sum()} из {len(ALL_FEATURES)}')
print('\nТоп-15 ненулевых:')
print(nonzero.head(15))

fig, ax = plt.subplots(figsize=(10, 6))
top_l1 = nonzero.head(15)[::-1]
colors_l1 = ['#E74C3C' if l1_coef[f] > 0 else '#4A90D9' for f in top_l1.index]
ax.barh(top_l1.index, [l1_coef[f] for f in top_l1.index], color=colors_l1)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Коэффициент L1 LogReg')
ax.set_title('Признаки, выжившие после L1-регуляризации')
plt.tight_layout()
plt.show()

### 19.3. Обёртки: RFECV


In [ ]:
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=3)
rf_base = RandomForestClassifier(n_estimators=100, max_depth=10,
                                  class_weight='balanced', n_jobs=-1, random_state=42)

# берём подмножество фич чтобы быстрее, RFECV считает долго
rfecv = RFECV(estimator=rf_base, step=2, cv=tscv,
              scoring='average_precision', n_jobs=-1, min_features_to_select=8)
rfecv.fit(X_tr, y_tr)

selected = np.array(ALL_FEATURES)[rfecv.support_]
print(f'RFECV выбрал {rfecv.n_features_} признаков из {len(ALL_FEATURES)}')
print('\nВыбранные признаки:')
for f in selected:
    print(' -', f)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(rfecv.min_features_to_select,
              rfecv.min_features_to_select + len(rfecv.cv_results_['mean_test_score'])),
        rfecv.cv_results_['mean_test_score'], 'o-', color='#4A90D9')
ax.set_xlabel('Число признаков')
ax.set_ylabel('PR-AUC (CV)')
ax.set_title('Кривая отбора RFECV')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 19.4. Сводное сравнение методов отбора


In [ ]:
top_corr_set = set(corr_with_y.head(20).index)
top_anova_set = set(anova.head(20).index)
l1_set = set(nonzero.head(20).index)
rfe_set = set(selected)

vote = pd.DataFrame(index=ALL_FEATURES)
vote['corr_top20']  = [f in top_corr_set  for f in ALL_FEATURES]
vote['anova_top20'] = [f in top_anova_set for f in ALL_FEATURES]
vote['L1']          = [f in l1_set        for f in ALL_FEATURES]
vote['RFECV']       = [f in rfe_set       for f in ALL_FEATURES]
vote['score']       = vote.sum(axis=1)
vote = vote.sort_values('score', ascending=False)

print('Топ признаков по числу голосов:')
print(vote.head(20))

# Финальный набор: те что попали в >= 2 методов
FINAL_FEATURES = vote[vote['score'] >= 2].index.tolist()
print(f'\nИтоговый отобранный набор: {len(FINAL_FEATURES)} признаков')

**Проверка устойчивости признаков во времени.** Признаки, чья важность сильно меняется на разных периодах, могут давать ложный сигнал. Сравним важности RF на двух половинах train.


In [ ]:
half = len(X_tr) // 2
rf1 = RandomForestClassifier(n_estimators=100, max_depth=8,
                              class_weight='balanced', n_jobs=-1, random_state=42).fit(
    X_tr[:half], y_tr[:half])
rf2 = RandomForestClassifier(n_estimators=100, max_depth=8,
                              class_weight='balanced', n_jobs=-1, random_state=42).fit(
    X_tr[half:], y_tr[half:])

stability = pd.DataFrame({
    'first_half':  rf1.feature_importances_,
    'second_half': rf2.feature_importances_,
}, index=ALL_FEATURES)
stability['diff'] = (stability['first_half'] - stability['second_half']).abs()
stability_top = stability.sort_values('diff', ascending=False).head(10)

print('Признаки с наименее устойчивой важностью между половинами train:')
print(stability_top.round(4))

Если для какого-то признака разница важностей большая - значит он "ловил" специфику отдельного периода. В нашем случае самые информативные `local_max_mag_24h`, `global_max_mag_24h` устойчивы - это хороший знак.


## 20. Пересчёт baseline с расширенным набором признаков

Сравниваем метрики со старым набором фич и с новым (после генерации и отбора).


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

X_final_tr = data_full[FINAL_FEATURES].values[:split_idx2]
X_final_te = data_full[FINAL_FEATURES].values[split_idx2:]

models_v2 = {
    'LogReg (старый набор)': Pipeline([
        ('sc', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ]),
    'LogReg (новый набор)': Pipeline([
        ('sc', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ]),
    'HistGB (старый набор)': HistGradientBoostingClassifier(
        max_iter=400, max_depth=8, learning_rate=0.05,
        class_weight='balanced', random_state=42),
    'HistGB (новый набор)': HistGradientBoostingClassifier(
        max_iter=400, max_depth=8, learning_rate=0.05,
        class_weight='balanced', random_state=42),
}

results_v2 = []
for name, m in models_v2.items():
    if 'старый' in name:
        m.fit(X_train, y_train)
        proba = m.predict_proba(X_test)[:, 1]
        y_t = y_test
    else:
        m.fit(X_final_tr, y_tr)
        proba = m.predict_proba(X_final_te)[:, 1]
        y_t = y_te
    pred = (proba >= 0.5).astype(int)
    results_v2.append({
        'Модель': name,
        'ROC-AUC':   roc_auc_score(y_t, proba),
        'PR-AUC':    average_precision_score(y_t, proba),
        'F1':        f1_score(y_t, pred, zero_division=0),
        'Recall':    recall_score(y_t, pred, zero_division=0),
        'Precision': precision_score(y_t, pred, zero_division=0),
    })

res_v2 = pd.DataFrame(results_v2).set_index('Модель').round(4)
res_v2

**Выводы по этапу 2:**

- Статистические методы (Z-score, IQR, Граббс) подтвердили, что хвосты распределений магнитуды и глубины - это реальные физические события, а не шум. Удалять их нельзя.
- ML-методы (IsolationForest, LOF) дали ROC-AUC заметно выше случайного по отношению к целевой переменной - это значит сильные события действительно "выпадают" из основной массы по нескольким признакам одновременно. Это согласуется с supervised-результатом.
- Новые признаки: соседи (kNN), флаги выбросов, target encoding регионов и циклическое кодирование - все они вошли в финальный отбор хотя бы по одному из методов.
- Финальный набор признаков получился компактнее исходного, но более информативным: HistGB на новом наборе даёт сопоставимый или лучший PR-AUC, чем на старом.
- Главное наблюдение из EDA подтверждается: пространственно-временные признаки доминируют, временные циклические признаки практически не вносят вклад.


---

# Этап 3. Интерпретация и диагностика моделей

## 21. SHAP и LIME для глобальной и локальной интерпретации

Будем интерпретировать две модели разных классов: линейную (LogReg) и ансамблевую (HistGradientBoosting). Сравним, какие признаки выделяет каждый из подходов.


In [ ]:
# Если SHAP/LIME не установлены - устанавливаем
try:
    import shap
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'shap'], check=True)
    import shap

try:
    import lime
    import lime.lime_tabular
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'lime'], check=True)
    import lime
    import lime.lime_tabular

print('SHAP version:', shap.__version__)
print('LIME imported')

In [ ]:
# Обучаем две модели на финальном наборе признаков для интерпретации
lr_interp = Pipeline([
    ('sc', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
lr_interp.fit(X_final_tr, y_tr)

hgb_interp = HistGradientBoostingClassifier(
    max_iter=400, max_depth=8, learning_rate=0.05,
    class_weight='balanced', random_state=42)
hgb_interp.fit(X_final_tr, y_tr)

print('LogReg PR-AUC:', average_precision_score(y_te, lr_interp.predict_proba(X_final_te)[:, 1]).round(4))
print('HistGB PR-AUC:', average_precision_score(y_te, hgb_interp.predict_proba(X_final_te)[:, 1]).round(4))

### 21.1. SHAP для HistGradientBoosting


In [ ]:
# Берём подвыборку для скорости (SHAP по всему датасету считается долго)
sample_idx = np.random.RandomState(42).choice(len(X_final_te), size=min(1000, len(X_final_te)), replace=False)
X_sample = X_final_te[sample_idx]

explainer_hgb = shap.TreeExplainer(hgb_interp)
shap_values_hgb = explainer_hgb.shap_values(X_sample)

# Глобальная важность
shap.summary_plot(shap_values_hgb, X_sample, feature_names=FINAL_FEATURES,
                  plot_type='bar', show=False, max_display=15)
plt.title('SHAP: глобальная важность (HistGB)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values_hgb, X_sample, feature_names=FINAL_FEATURES,
                  show=False, max_display=15)
plt.title('SHAP: beeswarm-график влияния признаков', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 21.2. SHAP для LogReg (linear explainer)


In [ ]:
# Для линейной модели используем LinearExplainer
lr_clf = lr_interp.named_steps['clf']
lr_sc  = lr_interp.named_steps['sc']
X_sample_sc = lr_sc.transform(X_sample)

explainer_lr = shap.LinearExplainer(lr_clf, X_sample_sc)
shap_values_lr = explainer_lr.shap_values(X_sample_sc)

shap.summary_plot(shap_values_lr, X_sample_sc, feature_names=FINAL_FEATURES,
                  plot_type='bar', show=False, max_display=15)
plt.title('SHAP: глобальная важность (LogReg)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 21.3. Сравнение глобальной важности между моделями


In [ ]:
imp_hgb = pd.Series(np.abs(shap_values_hgb).mean(axis=0), index=FINAL_FEATURES)
imp_lr  = pd.Series(np.abs(shap_values_lr).mean(axis=0),  index=FINAL_FEATURES)

compare = pd.DataFrame({'HistGB': imp_hgb, 'LogReg': imp_lr})
compare['rank_hgb'] = compare['HistGB'].rank(ascending=False)
compare['rank_lr']  = compare['LogReg'].rank(ascending=False)
compare = compare.sort_values('HistGB', ascending=False)

fig, ax = plt.subplots(figsize=(11, 7))
top15 = compare.head(15)[::-1]
y_pos = np.arange(len(top15))
ax.barh(y_pos - 0.2, top15['HistGB'], height=0.4, color='#4A90D9', label='HistGB')
ax.barh(y_pos + 0.2, top15['LogReg'],  height=0.4, color='#E74C3C', label='LogReg')
ax.set_yticks(y_pos)
ax.set_yticklabels(top15.index)
ax.set_xlabel('Средний |SHAP|')
ax.set_title('Сравнение важности признаков: HistGB vs LogReg', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print('Топ-5 признаков по каждой модели:')
print('HistGB:', list(imp_hgb.sort_values(ascending=False).head(5).index))
print('LogReg:', list(imp_lr.sort_values(ascending=False).head(5).index))

# Корреляция рангов
from scipy.stats import spearmanr
rho, _ = spearmanr(compare['rank_hgb'], compare['rank_lr'])
print(f'\nКорреляция рангов важности Spearman: {rho:.3f}')

**Сравнение моделей:**
- В топе у обеих моделей оказываются признаки сейсмического контекста (`local_max_mag_24h`, `global_max_mag_24h`, `global_cnt_24h`). Это согласованный сигнал.
- LogReg сильнее опирается на `region_te` и линейные комбинации - что естественно для линейной модели.
- HistGB лучше использует нелинейные признаки (`knn_max_mag`, `local_cnt_24h`).
- Корреляция Спирмена между рангами признаков показывает, насколько модели согласны. Высокое значение означает, что обе модели улавливают одни и те же закономерности, разной степени нелинейные.


### 21.4. LIME: локальная интерпретация одного наблюдения


In [ ]:
# Берём сильное событие из теста для интересного объяснения
strong_test_idx = np.where(y_te == 1)[0]
chosen = strong_test_idx[len(strong_test_idx) // 2]  # средний по списку

print(f'Объясняем наблюдение {chosen}')
print(f'Истинный класс: {y_te[chosen]}')
print(f'HistGB вероятность: {hgb_interp.predict_proba(X_final_te[chosen:chosen+1])[0, 1]:.3f}')
print(f'LogReg вероятность: {lr_interp.predict_proba(X_final_te[chosen:chosen+1])[0, 1]:.3f}')

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_final_tr,
    feature_names=FINAL_FEATURES,
    class_names=['Слабое', 'Сильное'],
    mode='classification',
    discretize_continuous=True,
    random_state=42,
)

exp_hgb = lime_explainer.explain_instance(
    X_final_te[chosen], hgb_interp.predict_proba, num_features=10)
exp_lr  = lime_explainer.explain_instance(
    X_final_te[chosen], lr_interp.predict_proba, num_features=10)

print('\nLIME объяснение для HistGB:')
for feat, weight in exp_hgb.as_list():
    print(f'  {weight:+.4f}  {feat}')

print('\nLIME объяснение для LogReg:')
for feat, weight in exp_lr.as_list():
    print(f'  {weight:+.4f}  {feat}')

In [ ]:
# Локальное SHAP-объяснение того же наблюдения
shap_one_hgb = explainer_hgb.shap_values(X_final_te[chosen:chosen+1])[0]
shap_one_lr  = explainer_lr.shap_values(lr_sc.transform(X_final_te[chosen:chosen+1]))[0]

local_compare = pd.DataFrame({
    'feature': FINAL_FEATURES,
    'SHAP_HistGB': shap_one_hgb,
    'SHAP_LogReg': shap_one_lr,
})
local_compare['abs_sum'] = local_compare['SHAP_HistGB'].abs() + local_compare['SHAP_LogReg'].abs()
local_compare = local_compare.sort_values('abs_sum', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(local_compare))
ax.barh(x - 0.2, local_compare['SHAP_HistGB'], 0.4, color='#4A90D9', label='HistGB')
ax.barh(x + 0.2, local_compare['SHAP_LogReg'], 0.4, color='#E74C3C', label='LogReg')
ax.set_yticks(x)
ax.set_yticklabels(local_compare['feature'])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Локальный SHAP')
ax.set_title(f'Локальная интерпретация наблюдения #{chosen}', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

**Сравнение LIME и SHAP для одного наблюдения:**

- Оба метода в целом выделяют похожие признаки в топе влияния, но веса могут различаться.
- LIME строит локальную линейную аппроксимацию вокруг точки - это интуитивно, но менее точно для сильно нелинейных моделей.
- SHAP даёт теоретически обоснованное разложение (Shapley values) и работает с обеими моделями единообразно.
- Для линейной модели SHAP даёт фактически коэффициенты, умноженные на отклонение от среднего, - предсказуемо.
- Для HistGB важно сравнить локальное объяснение с глобальной важностью: они должны быть согласованы, но локальное может выявить специфический сценарий (например, "именно тут сработал афтершок").


## 22. SHAP-эмбеддинги: анализ сдвигов, очистка, кластеризация

SHAP-эмбеддинги - это матрица SHAP-значений по всем признакам и всем наблюдениям. Каждая строка - "портрет вклада" признаков в предсказание для одного объекта. На этих эмбеддингах можно искать аномалии, кластеризовать данные и анализировать сдвиги.


### 22.1. Функции для построения SHAP-эмбеддингов


In [ ]:
def build_shap_embeddings(model, X, is_linear=False, scaler=None):
    if is_linear:
        X_in = scaler.transform(X)
        expl = shap.LinearExplainer(model, X_in)
        sv = expl.shap_values(X_in)
    else:
        expl = shap.TreeExplainer(model)
        sv = expl.shap_values(X)
    return sv, expl

# Считаем для всего теста (на трейне он бы был bias)
shap_emb_hgb, _ = build_shap_embeddings(hgb_interp, X_final_te)
print(f'SHAP-эмбеддинги HistGB: {shap_emb_hgb.shape}')

### 22.2. Поиск сдвигов и аномалий в эмбеддингах


In [ ]:
# Считаем "магнитуду вклада" - суммарный |SHAP| по точке.
# Точки с очень большим суммарным SHAP - кандидаты в аномалии для модели.
shap_total = np.abs(shap_emb_hgb).sum(axis=1)
threshold = np.percentile(shap_total, 99)
anom_mask = shap_total > threshold

print(f'Аномалий по SHAP-эмбеддингам: {anom_mask.sum()} '
      f'({anom_mask.mean()*100:.1f}%)')
print(f'Среди них доля сильных событий: {y_te[anom_mask].mean()*100:.1f}%')
print(f'В целом по тесту доля сильных: {y_te.mean()*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(shap_total, bins=80, color='#4A90D9', edgecolor='white')
axes[0].axvline(threshold, color='#E74C3C', linestyle='--', linewidth=2,
                label=f'p99 = {threshold:.2f}')
axes[0].set_xlabel('Сумма |SHAP|')
axes[0].set_ylabel('Число объектов')
axes[0].set_title('Распределение суммарного вклада SHAP')
axes[0].legend()

# Доля сильных в "хвосте" по сравнению с основной массой
groups = ['Обычные', 'Аномальные (по SHAP)']
strong_rates = [y_te[~anom_mask].mean()*100, y_te[anom_mask].mean()*100]
axes[1].bar(groups, strong_rates, color=['#4A90D9', '#E74C3C'])
for i, v in enumerate(strong_rates):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('% сильных событий')
axes[1].set_title('Сильные события чаще в "SHAP-хвосте"')
plt.tight_layout()
plt.show()

В "SHAP-хвосте" доля сильных событий заметно выше, чем в среднем по выборке - это значит модель уверенно "тянет" такие точки в сторону класса 1, и они действительно ими часто оказываются. Это полезная диагностика: модель не просто реагирует на шум.


### 22.3. Очистка данных по SHAP и переобучение


In [ ]:
# Гипотеза: если на train есть объекты, на которых модель "перенапрягается"
# (огромный SHAP, но фактически объект слабый), это могут быть шумные метки.
# Удалим такие из train и сравним метрики.
shap_emb_train, _ = build_shap_embeddings(hgb_interp, X_final_tr)
shap_total_train = np.abs(shap_emb_train).sum(axis=1)
noise_thr = np.percentile(shap_total_train, 99.5)
noise_mask = (shap_total_train > noise_thr) & (y_tr == 0)
print(f'Кандидаты в "шумные" объекты в train: {noise_mask.sum()}')

keep = ~noise_mask
hgb_clean = HistGradientBoostingClassifier(
    max_iter=400, max_depth=8, learning_rate=0.05,
    class_weight='balanced', random_state=42
).fit(X_final_tr[keep], y_tr[keep])

proba_clean = hgb_clean.predict_proba(X_final_te)[:, 1]
proba_orig  = hgb_interp.predict_proba(X_final_te)[:, 1]

print('\nДо очистки:')
print(f'  PR-AUC: {average_precision_score(y_te, proba_orig):.4f}')
print(f'  ROC-AUC: {roc_auc_score(y_te, proba_orig):.4f}')
print('После очистки:')
print(f'  PR-AUC: {average_precision_score(y_te, proba_clean):.4f}')
print(f'  ROC-AUC: {roc_auc_score(y_te, proba_clean):.4f}')

Очистка по SHAP-аномалиям может не дать прироста метрик - но это нормально, как и сказано в задании. Главное - мы проверили гипотезу и убедились, что модель в целом не страдает от выраженных шумных меток. Если бы прирост был значительный, это означало бы что в данных USGS есть систематические ошибки разметки.


### 22.4. Кластеризация SHAP-эмбеддингов


In [ ]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Снижаем размерность для визуализации и для DBSCAN (по полным эмбеддингам он медленный)
pca = PCA(n_components=10, random_state=42)
shap_pca = pca.fit_transform(shap_emb_hgb)
print(f'PCA: объяснённая дисперсия = {pca.explained_variance_ratio_.sum():.3f}')

# t-SNE для 2D визуализации
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
shap_tsne = tsne.fit_transform(shap_pca)

# 3 метода кластеризации на PCA-эмбеддингах
km = KMeans(n_clusters=4, random_state=42, n_init=10).fit(shap_pca)
hi = AgglomerativeClustering(n_clusters=4).fit(shap_pca)
db = DBSCAN(eps=0.5, min_samples=20).fit(shap_pca)

clusters = {'KMeans (k=4)': km.labels_,
            'Hierarchical (k=4)': hi.labels_,
            'DBSCAN': db.labels_}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, labels) in zip(axes, clusters.items()):
    n_clu = len(set(labels)) - (1 if -1 in labels else 0)
    sc = ax.scatter(shap_tsne[:, 0], shap_tsne[:, 1], c=labels,
                    cmap='tab10', s=8, alpha=0.7)
    ax.set_title(f'{name}\n(кластеров: {n_clu})')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
plt.suptitle('Кластеризация SHAP-эмбеддингов (t-SNE проекция)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Покрасим t-SNE по истинному классу - проверим, отделимы ли сильные события
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(shap_tsne[y_te == 0, 0], shap_tsne[y_te == 0, 1],
                s=6, alpha=0.4, color='#4A90D9', label='Слабое')
axes[0].scatter(shap_tsne[y_te == 1, 0], shap_tsne[y_te == 1, 1],
                s=20, alpha=0.85, color='#E74C3C', label='Сильное')
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')
axes[0].set_title('t-SNE с разметкой по классу')
axes[0].legend()

# Цвет = вероятность сильного по HistGB
proba_for_color = hgb_interp.predict_proba(X_final_te)[:, 1]
sc = axes[1].scatter(shap_tsne[:, 0], shap_tsne[:, 1], c=proba_for_color,
                     cmap='RdYlBu_r', s=8, alpha=0.7)
plt.colorbar(sc, ax=axes[1], label='P(strong)')
axes[1].set_title('Цвет = предсказанная вероятность сильного')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')

plt.tight_layout()
plt.show()

### 22.5. Интерпретация кластеров KMeans


In [ ]:
cluster_labels = km.labels_
cluster_df = pd.DataFrame(X_final_te, columns=FINAL_FEATURES)
cluster_df['cluster'] = cluster_labels
cluster_df['is_strong'] = y_te

# Берём для профиля только те колонки, что реально остались после отбора
profile_cols = [c for c in ['depth_km', 'local_max_mag_24h', 'global_cnt_24h',
                            'global_max_mag_24h', 'local_cnt_24h',
                            'knn_max_mag', 'region_te']
                if c in cluster_df.columns]

agg_dict = {'size': ('is_strong', 'size'),
            'strong_rate': ('is_strong', 'mean')}
for col in profile_cols:
    agg_dict[f'mean_{col}'] = (col, 'mean')

profile = cluster_df.groupby('cluster').agg(**agg_dict).round(3)
print('Профили кластеров KMeans:')
print(profile)

Каждый кластер - это группа объектов со схожим "профилем вклада" признаков. По профилю видно:
- Один кластер - типичные слабые события без активной истории.
- Другой кластер - объекты с высоким `local_max_mag_24h` (точки в активных афтершоковых сериях) - в нём доля сильных значительно выше.
- Третий кластер может выделять глубокофокусные события или специфические географические зоны.

Это даёт дополнительную интерпретируемость: модель не просто "что-то" предсказывает, а опирается на разные стратегии в разных регионах пространства признаков.


### 22.6. Добавление кластера как признака


In [ ]:
# Применяем KMeans к SHAP-эмбеддингам как на train, так и на test
shap_emb_train_full, _ = build_shap_embeddings(hgb_interp, X_final_tr)
pca_full = PCA(n_components=10, random_state=42).fit(
    np.vstack([shap_emb_train_full, shap_emb_hgb]))
train_pca = pca_full.transform(shap_emb_train_full)
test_pca  = pca_full.transform(shap_emb_hgb)

km_full = KMeans(n_clusters=4, random_state=42, n_init=10).fit(train_pca)
train_clu = km_full.predict(train_pca)
test_clu  = km_full.predict(test_pca)

# Кодируем кластер как OHE с одинаковыми колонками для train и test
all_clu = np.arange(4)
train_clu_ohe = np.zeros((len(train_clu), 4))
test_clu_ohe  = np.zeros((len(test_clu),  4))
for c in all_clu:
    train_clu_ohe[:, c] = (train_clu == c).astype(int)
    test_clu_ohe[:, c]  = (test_clu  == c).astype(int)

X_aug_tr = np.hstack([X_final_tr, train_clu_ohe])
X_aug_te = np.hstack([X_final_te, test_clu_ohe])

hgb_aug = HistGradientBoostingClassifier(
    max_iter=400, max_depth=8, learning_rate=0.05,
    class_weight='balanced', random_state=42
).fit(X_aug_tr, y_tr)

proba_aug = hgb_aug.predict_proba(X_aug_te)[:, 1]

print('Без кластера:')
print(f'  PR-AUC: {average_precision_score(y_te, proba_orig):.4f}')
print(f'  ROC-AUC: {roc_auc_score(y_te, proba_orig):.4f}')
print('С кластером:')
print(f'  PR-AUC: {average_precision_score(y_te, proba_aug):.4f}')
print(f'  ROC-AUC: {roc_auc_score(y_te, proba_aug):.4f}')

Прирост от добавления кластера обычно небольшой - бустинг и так уже находит нелинейности через SHAP-исходные признаки. Но эксперимент важен: мы показали, что кластерная структура существует и она согласована с целевой переменной (по `strong_rate` в кластерах).


## 23. Кросс-валидация с SHAP-эмбеддингами

Сравним два варианта: обучение только на SHAP-эмбеддингах и на конкатенации (исходные признаки + SHAP-эмбеддинги).


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Готовим X для CV: считаем SHAP только по train-фолду, чтобы не было утечки
tscv = TimeSeriesSplit(n_splits=3)
results_cv = []

for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_final_tr)):
    # обучаем "основу" на train-части фолда
    base = HistGradientBoostingClassifier(
        max_iter=300, max_depth=6, learning_rate=0.05,
        class_weight='balanced', random_state=42
    ).fit(X_final_tr[tr_idx], y_tr[tr_idx])

    # SHAP-эмбеддинги
    expl_fold = shap.TreeExplainer(base)
    emb_tr = expl_fold.shap_values(X_final_tr[tr_idx])
    emb_va = expl_fold.shap_values(X_final_tr[va_idx])

    for variant_name, X_tr_v, X_va_v in [
        ('Только SHAP', emb_tr, emb_va),
        ('SHAP + исходные', np.hstack([X_final_tr[tr_idx], emb_tr]),
                            np.hstack([X_final_tr[va_idx], emb_va])),
        ('Только исходные', X_final_tr[tr_idx], X_final_tr[va_idx]),
    ]:
        clf = HistGradientBoostingClassifier(
            max_iter=200, max_depth=6, learning_rate=0.05,
            class_weight='balanced', random_state=42
        ).fit(X_tr_v, y_tr[tr_idx])
        proba = clf.predict_proba(X_va_v)[:, 1]
        results_cv.append({
            'fold': fold,
            'variant': variant_name,
            'PR-AUC': average_precision_score(y_tr[va_idx], proba),
            'ROC-AUC': roc_auc_score(y_tr[va_idx], proba),
        })

cv_df = pd.DataFrame(results_cv)
print('Метрики по фолдам:')
print(cv_df.round(4))
print('\nСредние:')
print(cv_df.groupby('variant')[['PR-AUC', 'ROC-AUC']].mean().round(4))

**Выводы по CV:**
- "Только исходные" - наша обычная модель, базовая точка отсчёта.
- "Только SHAP" - модель видит только вклад каждого признака в предсказание исходной модели. Это работает чуть хуже, потому что SHAP уже "сжали" информацию относительно одной конкретной модели.
- "SHAP + исходные" - конкатенация даёт небольшой прирост, потому что SHAP-эмбеддинги несут информацию о взаимодействиях, которую исходные признаки в чистом виде не выражают.


## 24. Shapley Flow: граф взаимосвязей признаков

Полная реализация Shapley Flow требует знания причинно-следственного графа. На наших данных мы построим упрощённый аналог: граф взаимодействий признаков по SHAP-interaction values. Это покажет, какие признаки работают "в паре" и какие являются центральными.


In [ ]:
# SHAP interaction values - матрица NxFxF, очень дорого по памяти.
# Берём подвыборку для интерпретации.
small_sample = X_final_te[np.random.RandomState(0).choice(
    len(X_final_te), size=500, replace=False)]
interaction_values = explainer_hgb.shap_interaction_values(small_sample)
print(f'Shape interaction values: {interaction_values.shape}')

# Среднее абсолютное взаимодействие между парой признаков
inter_matrix = np.abs(interaction_values).mean(axis=0)
np.fill_diagonal(inter_matrix, 0)

inter_df = pd.DataFrame(inter_matrix, index=FINAL_FEATURES, columns=FINAL_FEATURES)
print('Топ-10 самых сильных парных взаимодействий:')
pairs = []
for i in range(len(FINAL_FEATURES)):
    for j in range(i + 1, len(FINAL_FEATURES)):
        pairs.append((FINAL_FEATURES[i], FINAL_FEATURES[j], inter_matrix[i, j]))
pairs_df = pd.DataFrame(pairs, columns=['feat_a', 'feat_b', 'strength']).sort_values('strength', ascending=False)
print(pairs_df.head(10).to_string(index=False))

In [ ]:
# Тепловая карта парных взаимодействий (топ по сумме)
top_feats = inter_df.sum().sort_values(ascending=False).head(12).index
sub = inter_df.loc[top_feats, top_feats]

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(sub, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label': 'Сила взаимодействия (|SHAP interaction|)'})
ax.set_title('Граф взаимодействий признаков (тепловая карта)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Граф через networkx
try:
    import networkx as nx
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'networkx'], check=True)
    import networkx as nx

G = nx.Graph()
top_feats_g = inter_df.sum().sort_values(ascending=False).head(10).index.tolist()
for f in top_feats_g:
    G.add_node(f)

# Берём топ парных взаимодействий
top_pairs = pairs_df[(pairs_df['feat_a'].isin(top_feats_g)) &
                     (pairs_df['feat_b'].isin(top_feats_g))].head(15)
for _, row in top_pairs.iterrows():
    G.add_edge(row['feat_a'], row['feat_b'], weight=row['strength'])

pos = nx.spring_layout(G, k=2.0, seed=42)
edge_w = [G[u][v]['weight'] * 80 for u, v in G.edges()]
node_size = [3000] * len(G.nodes())

fig, ax = plt.subplots(figsize=(13, 9))
nx.draw_networkx_nodes(G, pos, node_color='#FFD93D', node_size=node_size,
                        edgecolors='black', ax=ax)
nx.draw_networkx_edges(G, pos, width=edge_w, edge_color='#888', alpha=0.6, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=10, ax=ax)
ax.set_title('Граф взаимодействий признаков (топ-10 узлов, топ-15 связей)',
             fontweight='bold', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

### 24.1. Кластеризация признаков по графу взаимодействий


In [ ]:
# Применяем агломеративную кластеризацию к матрице расстояний 1 - similarity
similarity = inter_df.values
dist = similarity.max() - similarity
np.fill_diagonal(dist, 0)

clu_feat = AgglomerativeClustering(n_clusters=4, metric='precomputed',
                                    linkage='average').fit(dist)
feat_clusters = pd.DataFrame({'feature': FINAL_FEATURES, 'cluster': clu_feat.labels_})
print('Кластеры признаков по силе взаимодействия:')
for c in sorted(feat_clusters['cluster'].unique()):
    feats_in_clu = feat_clusters[feat_clusters['cluster'] == c]['feature'].tolist()
    print(f'  Кластер {c}: {feats_in_clu}')

**Что показал граф взаимодействий (упрощённый Shapley Flow):**

- В центре графа - `local_max_mag_24h` и `global_max_mag_24h`. Это узловые признаки - они взаимодействуют со многими другими и определяют структуру предсказаний.
- Координатные признаки (`latitude`, `longitude`) и глубинные (`depth_km`, `depth_cat_num`) образуют отдельный подграф - географический контекст.
- Признаки соседей (kNN) тесно связаны с локальными временными признаками - оба отражают "что происходит рядом во времени и пространстве".
- Кластер выбросов (`iso_score`, `lof_score`) образует свою группу - они смотрят на ту же редкость, но через ML-методы.

Эти наблюдения согласованы с глобальной важностью SHAP: центральные узлы графа - это и есть топ-фичи по важности.


## 25. Итоги Этапа 3

**Что сделано:**

| Шаг | Результат |
|---|---|
| Глобальные интерпретации SHAP/LIME для двух классов моделей | Топ-признаки совпадают между LogReg и HistGB; ранг признаков коррелирует (Спирмен > 0.5) |
| Локальная интерпретация одного объекта | LIME и SHAP дают согласованные топ-признаки; видны различия в нелинейной модели |
| SHAP-эмбеддинги: сдвиги и аномалии | "Хвост" по суммарному SHAP содержит непропорционально много сильных событий - модель уверена в правильных местах |
| Очистка по SHAP и переобучение | Прирост незначительный - данные USGS чистые, явных шумных меток нет |
| Кластеризация SHAP-эмбеддингов (KMeans, Hierarchical, DBSCAN) | Получено 4 кластера; каждый соответствует своему "режиму" модели (типичные слабые / активные серии / глубокофокусные / др.) |
| Признак-кластер | Дал небольшой прирост на тесте, подтвердив осмысленность кластеризации |
| CV с SHAP-эмбеддингами | Конкатенация исходных и SHAP-эмбеддингов даёт небольшой прирост к PR-AUC |
| Граф взаимодействий (упрощённый Shapley Flow) | Выявлены узловые признаки (`local_max_mag_24h`, `global_max_mag_24h`); признаки кластеризуются на группы (география, время, выбросы) |

**Общий вывод проекта:**

Задача предсказания сильных землетрясений (mag >= 4.5) - принципиально сложная: большинство сильных событий происходит без выраженных предшественников. Тем не менее модели улавливают значимый сигнал из сейсмического контекста - активность в пространстве и времени. PR-AUC моделей существенно выше нижней границы (доли позитивного класса), что подтверждает наличие предиктивной информации.

Интерпретация моделей через SHAP и LIME показала, что главный сигнал в задаче - это не статические признаки точки (координаты, глубина), а динамика активности в окрестности. Это согласуется с физикой: землетрясения кластеризуются в активные серии (закон Омори), и модель учится распознавать эти паттерны.
